# DataMate 2026 — Evaluate Ornith 1.5 (9B) + OpenCode trên Kaggle T4 ×2

Notebook này tái lập luồng agent **Ornith 1.5 9B + OpenCode 1.19.5 + Ollama 0.40.7** trên Kaggle **GPU T4 ×2**, hỗ trợ 2 chế độ:
1. **`eval_agent` (Chế độ chính):** cấp `read`, `inspect_table`, `aggregate` và **`run_python`** (bọc `run_python_in_sandbox`) để giải các bài ngoài CSV `count_distinct` (tổng doanh thu, tỷ lệ %, bảng/danh sách, XLSX, SQLite).
2. **`structured_only` (Chế độ đối chứng local):** chỉ cấp `read`, `inspect_table`, `aggregate` (`count_distinct` trên CSV).

---

## Quy trình chạy an toàn 2 phiên (chống rò rỉ đáp án qua `run_python`)

`run_python_in_sandbox` chạy tiến trình Python với quyền đọc filesystem của notebook. Vì vậy:
- **Phiên 1 (`EXECUTION_STAGE = "inference"`):** CHỈ attach dataset câu hỏi + gói dữ liệu shop (`questions.jsonl`, `rulebook.md`, `calendar.csv`, `data/<business_id>/`) — **tuyệt đối KHÔNG attach `answers.jsonl` hay `eval_gold/`** khi dùng `eval_agent` (`run_python`). Mặc định notebook chạy hai câu smoke: đếm đơn `q_5c19a3193881` bằng `aggregate` và tra giá `q_01a28f2233bd` bằng `run_python`. Khi cả hai smoke pass mới bật `RUN_BATCH_EVAL = True`.
- **Phiên 2 (`EXECUTION_STAGE = "score_only"`):** attach `predictions.jsonl` đã khóa từ Phiên 1 cùng `answers.jsonl` / `eval_gold/unseen_gold_50.jsonl` để chấm điểm PA chính thức bằng `src/eval/scorer.py`.

## Kaggle Inputs cần attach (hoặc tự động tải từ Hugging Face)

Notebook hỗ trợ **tự động tải toàn bộ Input cần thiết từ Hugging Face** (`Crinkk/Zalo-challenge` — gói riêng biệt `ornith_opencode_inference_bundle.zip` không chứa đáp án cho Phiên 1 và `ornith_opencode_reference_bundle.zip` cho Phiên 2; cấu hình `HF_TOKEN` trong Kaggle Secrets nếu muốn tải từ repo gated/tăng tốc, hoặc tải public trực tiếp từ `dataset:Crinkk/Zalo-challenge`), **hoặc** dùng trực tiếp Kaggle Inputs gắn sẵn:

1. **Model bundle (`ornith15-datamate-pilot-8k:latest`)**:
   - Parent: `ornith-1.5:9b`, GGUF `Q4_K_M`, `9.0B` (`6,550,813,701` bytes).
   - Base tag digest: `57ad8bec8adf0a704a07a26adb2abf9edae310ba0fd3cef575d02dac1af69bb5`
   - Blobs trong `ollama show --modelfile`:
     - `852922174ee4f76621df26105333f1dfe2171cdfb60ebe5a4b013836681a8a77`
     - `2635637dd72b4d39a97331373a73366b69a0a6a14151b2a1da0a97b311e2461f`
   - Chat template SHA-256 (`/api/show`): `f55f52930aa8bf44ab5cb85f99370fcc3c56e9a85640b812086d5330bce5d86b`
2. **Repo + Structured Tools Bundle**:
   - Chứa `src/eval/scorer.py`, `src/schemas.py`, và snapshot `src/` của Khanh Linh tại `efee164a857736181f9dcd0ebf023f47b11e8844` (`opencode-structured/src/` hoặc repo git còn commit `efee164a`).
3. **Data Split Inference (không kèm `answers.jsonl`/`eval_gold/` ở Phiên 1)**.
4. **Runtime binaries (khuyến nghị khi offline / tránh chờ npm ~85s)**:
   - `ollama` `0.40.7` (`ollama-linux-amd64.tar.zst` hoặc binary đã giải nén + `lib/ollama`), `opencode` `1.19.5`, và `node_modules` pre-staged cho custom tools của OpenCode.


In [ ]:
# ==============================================================================
# CELL 1: CONFIG, 50-QUESTION MANIFEST, AUTO-DOWNLOAD HF & PREFLIGHT
# ==============================================================================
import hashlib
import io
import json
import os
import shutil
import subprocess
import sys
import tarfile
import zipfile
from datetime import datetime
from pathlib import Path

# ------------------------------------------------------------------------------
# 1. Cấu hình chính (Chỉnh sửa tại đây trước khi Run All)
# ------------------------------------------------------------------------------
EXECUTION_STAGE = os.environ.get("DATAMATE_STAGE", "inference")  # "inference" | "score_only"
SPLIT = os.environ.get("DATAMATE_SPLIT", "train")  # "train" | "public_test"
TOOL_MODE = os.environ.get("DATAMATE_TOOL_MODE", "eval_agent")  # "eval_agent" | "structured_only" | "auto_readme_schema"

# Hugging Face Input Bundles (tách riêng bản inference không đáp án và bản score_only)
HF_DATASET_REPO = os.environ.get("DATAMATE_HF_REPO", "Crinkk/Zalo-challenge")
HF_INFERENCE_BUNDLE_PATH = "ornith_opencode_kaggle/ornith_opencode_inference_bundle_v3.zip"
HF_REFERENCE_BUNDLE_PATH = "ornith_opencode_kaggle/ornith_opencode_reference_bundle.zip"
HF_OLLAMA_BLOBS_PREFIX = "ornith_opencode_kaggle/ollama_blobs"

# An toàn mặc định: chạy đúng 1 câu smoke biz_125 trước; chỉ bật batch khi smoke đã qua
RUN_SMOKE_TESTS = True
SMOKE_SINGLE_QUESTION_ONLY = False  # True: chỉ chạy Smoke 1; False: chạy thêm Smoke 2
FAIL_FAST_ON_SMOKE_ERROR = True
REQUIRE_SMOKE_PASS_FOR_BATCH = True
RUN_BATCH_EVAL = False  # Đổi thành True sau khi smoke test trên 1x T4 đã PASS
RUN_CONTROL_CSV_COUNT_DISTINCT = True  # Khi RUN_BATCH_EVAL=True, chạy kèm 12 câu CSV count_distinct đối chứng bằng structured_only
ENABLE_DUAL_WORKER_BENCHMARK = False  # Bật để chạy thêm probe hai GPU sau smoke

# Guard chống lộ reference trên filesystem khi run_python được bật trong Phiên inference
ALLOW_EXPOSED_REFERENCE_IN_RUN_PYTHON = os.environ.get("DATAMATE_ALLOW_EXPOSED_REF", "0") == "1"
SCORING_INPUT_RUN_DIR = os.environ.get("DATAMATE_SCORING_RUN_DIR", "")  # Dùng cho EXECUTION_STAGE == "score_only"

TEMPERATURES = [0.0, 0.4]
PARALLEL_SLOTS_PER_GPU = 1  # Ollama 0.40.7 giới hạn họ qwen35 ở một request đồng thời mỗi daemon
NUM_WORKERS = 2  # Một daemon và một lượt giải đồng thời trên mỗi T4

# Tham số model & OpenCode đã kiểm chứng ở local
BASE_MODEL_TAG = "ornith15-datamate-pilot-8k:latest"
EXPECTED_OPENCODE_VERSION = "1.19.5"
EXPECTED_OLLAMA_VERSION = "0.40.7"
EXPECTED_SNAPSHOT_COMMIT = "efee164a857736181f9dcd0ebf023f47b11e8844"
EXPECTED_MODEL_DIGEST = "57ad8bec8adf0a704a07a26adb2abf9edae310ba0fd3cef575d02dac1af69bb5"
EXPECTED_BLOB_SHA256S = [
    "852922174ee4f76621df26105333f1dfe2171cdfb60ebe5a4b013836681a8a77",
    "2635637dd72b4d39a97331373a73366b69a0a6a14151b2a1da0a97b311e2461f",
]
EXPECTED_TEMPLATE_SHA256 = "f55f52930aa8bf44ab5cb85f99370fcc3c56e9a85640b812086d5330bce5d86b"
EXPECTED_GGUF_BYTES = 6550813701

NUM_CTX = 65536
SEED = 42
NUM_BATCH = 128  # Giữ nguyên tham số pilot để quỹ đạo sinh token tại temp=0.0 khớp tuyệt đối
PRESENCE_PENALTY = 0
MAX_OUTPUT_TOKENS = 4096
MAX_STEPS = 10
MAX_STEPS_EVAL_AGENT = 14
TOOL_MAX_BYTES = 1000
TOOL_MAX_BYTES_EVAL_AGENT = 8000
RUN_PYTHON_TIMEOUT_SEC = 90
OPENCODE_TOOL_TIMEOUT_MS = 180000
PER_QUESTION_TIMEOUT_SEC = 600

# Smoke test IDs (chỉ dùng đáp án đối chứng SAU khi inference xong)
SMOKE_AGGREGATE_QID = "q_5c19a3193881"  # biz_125 count_distinct -> 41
SMOKE_RUN_PYTHON_QID = "q_01a28f2233bd"  # biz_112, giá bán một ly soda chanh dây (M)
SMOKE_RUN_PYTHON_EXPECTED = ("number", 37000)

# Manifest 50 câu cố định (khớp eval_gold/unseen_gold_50.jsonl: 16 shop, 11 CSV + 37 XLSX + 2 SQLite, đủ 8 type)
MANIFEST_50_IDS = [
    "q_019ec94d9799", "q_01a28f2233bd", "q_02a70ea566f1", "q_0c20dcf7fc9f", "q_132add5002d2",
    "q_153f4e995558", "q_1b119bdd5d28", "q_1f99d93acad3", "q_20aa992c0d3f", "q_21c12147e6d6",
    "q_22f59bab1075", "q_2a0e86444947", "q_2dc155c6526d", "q_2ea34572153a", "q_32c6bd8dd5ce",
    "q_338ae382671b", "q_360f8795ce6c", "q_3b5c9002b013", "q_43df08e6cce8", "q_554d736852a5",
    "q_583e4896e764", "q_58b4641d44b5", "q_5ade9d699c7f", "q_5c5ff524d0d9", "q_67297f4bdfc2",
    "q_6a5401d8e030", "q_6bcfadf41171", "q_725f54f95253", "q_7428165a9e0a", "q_77de8b009478",
    "q_80f40d2055e8", "q_85769fb47c38", "q_8c4380293447", "q_95099c4916a8", "q_97378cee7c0e",
    "q_9b1dce77c710", "q_a264d94a0492", "q_a569fbc6b9fc", "q_aacf9543a235", "q_bf91db6d5335",
    "q_c0c508d7fdfd", "q_d401c0884402", "q_da1e65bdc743", "q_e0aef192cb11", "q_e311e396f7f0",
    "q_e61ebdabf51c", "q_e7837d5638fb", "q_e9930d87abdd", "q_fca74e8bad45", "q_fcafb39ec35f",
]

# Nhóm 12 câu CSV count_distinct từ 4 shop holdout (biz_111, biz_114, biz_127, biz_129) để đối chứng structured_only
CSV_COUNT_DISTINCT_HOLDOUT_IDS = [
    "q_0c0a18470dcf", "q_1dcd312143bb", "q_26fb21e617a3",
    "q_81d119a55e5c", "q_93e78b3a3a98", "q_9a9d18b10b6e",
    "q_1c09918a7b72", "q_4909c1c5b936", "q_50041d4395a2",
    "q_1ef86d51bc58", "q_3094bfe79729", "q_73d64040f361",
]

MANUAL_PATHS = {
    "repo_dir": os.environ.get("DATAMATE_REPO_DIR", ""),
    "data_root": os.environ.get("DATAMATE_SPLIT_ROOT", ""),
    "model_dir": os.environ.get("DATAMATE_MODEL_INPUT_DIR", ""),
    "structured_dir": os.environ.get("DATAMATE_STRUCTURED_DIR", ""),
    "runtime_bin_dir": os.environ.get("DATAMATE_RUNTIME_BIN_DIR", ""),
    "reference_file": os.environ.get("DATAMATE_REFERENCE_FILE", ""),
}

# ------------------------------------------------------------------------------
# 2. Thiết lập thư mục làm việc & Auto-discovery các Kaggle Inputs
# ------------------------------------------------------------------------------
IS_KAGGLE = Path("/kaggle/working").exists()
WORK_ROOT = Path("/kaggle/working") if IS_KAGGLE else (Path.cwd() / "tmp" / "kaggle_ornith_eval_work")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = WORK_ROOT / f"ornith_opencode_{SPLIT}_{TOOL_MODE}_{RUN_TIMESTAMP}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUN_DIR: {RUN_DIR} (IS_KAGGLE={IS_KAGGLE}, EXECUTION_STAGE={EXECUTION_STAGE})")


def remove_tree_safe(target: Path) -> None:
    '''Xóa thư mục chứa file read-only (0o444) an toàn trên cả Linux và Windows.'''
    if not target.exists():
        return
    for root, dirs, files in os.walk(target):
        for d in dirs:
            try:
                os.chmod(Path(root) / d, 0o755)
            except OSError:
                pass
        for f in files:
            try:
                os.chmod(Path(root) / f, 0o644)
            except OSError:
                pass
    shutil.rmtree(target)


def check_gpus():
    try:
        proc = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, check=False, timeout=15)
        lines = [line.strip() for line in proc.stdout.splitlines() if line.strip()]
        print("nvidia-smi -L:")
        for line in lines:
            print("  ", line)
        return lines
    except Exception as exc:
        print(f"[WARN] Không chạy được nvidia-smi -L: {exc}")
        return []


GPU_LINES = check_gpus()
GPU_COUNT = len(GPU_LINES)

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
SEARCH_ROOTS = []
if KAGGLE_INPUT_ROOT.exists():
    SEARCH_ROOTS.extend(sorted(p for p in KAGGLE_INPUT_ROOT.iterdir() if p.is_dir()))
LOCAL_CWD = Path.cwd()
for cand_local in [LOCAL_CWD, LOCAL_CWD.parent]:
    if (cand_local / "src" / "eval" / "scorer.py").exists() and cand_local not in SEARCH_ROOTS:
        SEARCH_ROOTS.append(cand_local)


def find_candidate_dir(predicate, manual_val=""):
    if manual_val:
        mp = Path(manual_val)
        if mp.exists() and predicate(mp):
            return mp
    for root in SEARCH_ROOTS:
        if predicate(root):
            return root
        for depth_glob in ["*", "*/*"]:
            for sub in sorted(root.glob(depth_glob)):
                if sub.is_dir() and predicate(sub):
                    return sub
    return None


def find_candidate_file(predicate, manual_val=""):
    if manual_val:
        mp = Path(manual_val)
        if mp.is_file() and predicate(mp):
            return mp
    for root in SEARCH_ROOTS:
        for depth_glob in ["*", "*/*", "*/*/*"]:
            for sub in sorted(root.glob(depth_glob)):
                if sub.is_file() and predicate(sub):
                    return sub
    return None


def scan_exposed_reference_files(roots: list[Path], extra_dirs: list[Path | None] = ()) -> list[Path]:
    '''Phát hiện file đáp án chuẩn trên filesystem để chặn rủi ro run_python đọc nhầm đáp án.'''
    found: dict[str, Path] = {}
    scan_dirs: list[Path] = []
    for r in [*roots, *extra_dirs]:
        if r and r.exists():
            scan_dirs.append(r)
    for base in scan_dirs:
        for rel_glob in [
            "answers.jsonl",
            "*/answers.jsonl",
            "*/*/answers.jsonl",
            "data/*/answers.jsonl",
            "*gold*.jsonl",
            "eval_gold/*.jsonl",
            "*/*gold*.jsonl",
            "reference*.jsonl",
            "*/reference*.jsonl",
        ]:
            for cand in base.glob(rel_glob):
                if cand.is_file() and ".git" not in cand.parts:
                    found[str(cand.resolve())] = cand.resolve()
    return [found[k] for k in sorted(found)]


REPO_DIR = find_candidate_dir(
    lambda p: (p / "src" / "eval" / "scorer.py").is_file() and (p / "src" / "schemas.py").is_file(),
    MANUAL_PATHS["repo_dir"],
)
SPLIT_DIR = find_candidate_dir(
    lambda p: (p / "questions.jsonl").is_file() and (p / "rulebook.md").is_file() and (p / "data").is_dir(),
    MANUAL_PATHS["data_root"] or (str(REPO_DIR / "data" / SPLIT) if REPO_DIR and (REPO_DIR / "data" / SPLIT / "questions.jsonl").is_file() else ""),
)
MODEL_INPUT_DIR = find_candidate_dir(
    lambda p: (((p / "manifests").is_dir() or (p / "manifests-v2").is_dir()) and (p / "blobs").is_dir()) or any(p.glob("*.gguf")),
    MANUAL_PATHS["model_dir"],
)
STRUCTURED_INPUT_DIR = find_candidate_dir(
    lambda p: (p / "bridge.py").is_file() and (p / "run.js").is_file() and (p / "tools").is_dir(),
    MANUAL_PATHS["structured_dir"] or (str(REPO_DIR / "tmp" / "opencode-structured") if REPO_DIR and (REPO_DIR / "tmp" / "opencode-structured" / "bridge.py").is_file() else ""),
)
RUNTIME_BIN_DIR = find_candidate_dir(
    lambda p: (p / "ollama").exists() or (p / "opencode").exists() or (p / "node_modules").is_dir() or any(p.glob("ollama*")) or any(p.glob("opencode*")),
    MANUAL_PATHS["runtime_bin_dir"],
)


def _resolve_hf_token() -> str:
    tok = os.environ.get("HF_TOKEN", "").strip()
    if tok:
        return tok
    try:
        from kaggle_secrets import UserSecretsClient
        tok = (UserSecretsClient().get_secret("HF_TOKEN") or "").strip()
        if tok:
            return tok
    except Exception:
        pass
    return ""


HF_TOKEN_RUNTIME = _resolve_hf_token()


def download_from_hf(filename: str, local_dir: Path | None = None) -> Path:
    '''Tải file từ HF: ưu tiên repo model nếu có HF_TOKEN, fallback sang dataset công khai với token=False.'''
    from huggingface_hub import hf_hub_download

    attempts = []
    tok = HF_TOKEN_RUNTIME or os.environ.get("HF_TOKEN", "").strip()
    if tok:
        attempts.append(("model", tok))
        attempts.append(("dataset", tok))
    attempts.append(("dataset", False))

    last_exc = None
    for repo_type, use_tok in attempts:
        try:
            kwargs = dict(
                repo_id=HF_DATASET_REPO,
                filename=filename,
                repo_type=repo_type,
                token=use_tok,
            )
            if local_dir is not None:
                kwargs["local_dir"] = str(local_dir)
            return Path(hf_hub_download(**kwargs))
        except Exception as exc:
            last_exc = exc
    raise RuntimeError(f"Không tải được {filename} từ Hugging Face ({HF_DATASET_REPO}): {last_exc}")


def _extract_zip_clean(zip_path: Path, target_dir: Path) -> Path:
    remove_tree_safe(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(target_dir)
    return target_dir


HF_STAGE2_REF_DIR = WORK_ROOT / "hf_reference_stage2"
if EXECUTION_STAGE == "inference" and HF_STAGE2_REF_DIR.exists():
    remove_tree_safe(HF_STAGE2_REF_DIR)

HF_INFERENCE_BUNDLE_DIR = WORK_ROOT / "hf_inference_bundle"
if os.environ.get("DATAMATE_DISABLE_HF_AUTO_DOWNLOAD", "0") != "1":
    need_inf_bundle = (
        (EXECUTION_STAGE == "inference" and (not REPO_DIR or not SPLIT_DIR or not MODEL_INPUT_DIR))
        or (EXECUTION_STAGE == "score_only" and (not REPO_DIR or not SPLIT_DIR))
    )
    if need_inf_bundle:
        print(f"Đang tải inference bundle từ Hugging Face ({HF_DATASET_REPO}/{HF_INFERENCE_BUNDLE_PATH})...")
        _inf_zip = download_from_hf(HF_INFERENCE_BUNDLE_PATH)
        _extract_zip_clean(_inf_zip, HF_INFERENCE_BUNDLE_DIR)
        if not REPO_DIR and (HF_INFERENCE_BUNDLE_DIR / "src" / "eval" / "scorer.py").is_file():
            REPO_DIR = HF_INFERENCE_BUNDLE_DIR
        if not SPLIT_DIR and (HF_INFERENCE_BUNDLE_DIR / "data" / SPLIT / "questions.jsonl").is_file():
            SPLIT_DIR = HF_INFERENCE_BUNDLE_DIR / "data" / SPLIT
        if not STRUCTURED_INPUT_DIR and (HF_INFERENCE_BUNDLE_DIR / "tmp" / "opencode-structured" / "bridge.py").is_file():
            STRUCTURED_INPUT_DIR = HF_INFERENCE_BUNDLE_DIR / "tmp" / "opencode-structured"
        if not MODEL_INPUT_DIR and (HF_INFERENCE_BUNDLE_DIR / "ollama_pilot_overlay" / "blobs").is_dir():
            MODEL_INPUT_DIR = HF_INFERENCE_BUNDLE_DIR / "ollama_pilot_overlay"

# ------------------------------------------------------------------------------
# 3. Chuẩn bị Structured Tool Runtime & Snapshot src/ (efee164a)
# ------------------------------------------------------------------------------
OPENCODE_RUNTIME_DIR = WORK_ROOT / "opencode_structured_runtime"
OPENCODE_RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
(OPENCODE_RUNTIME_DIR / "tools").mkdir(parents=True, exist_ok=True)

EMBEDDED_RUN_JS = 'export function run(toolName, args) {\n  const pythonBin = process.env.DATAMATE_PYTHON_BIN || "python3";\n  const bridgePath = process.env.DATAMATE_BRIDGE_PATH || "/opt/datamate-structured/bridge.py";\n  const timeoutMs = Number(process.env.DATAMATE_TOOL_TIMEOUT_MS || "30000");\n\n  const result = Bun.spawnSync([\n    pythonBin, bridgePath, toolName, JSON.stringify(args),\n  ], { timeout: timeoutMs, env: process.env });\n\n  if (result.exitCode !== 0) {\n    return JSON.stringify({ error: result.stderr.toString().trim() || "Python tool failed" });\n  }\n  return result.stdout.toString().trim();\n}\n'
EMBEDDED_INSPECT_TABLE_JS = 'import { run } from "../run.js";\n\nexport default {\n  description: "Tìm bảng CSV, kiểm tra tên cột hoặc xem giá trị thực tế trong một cột.",\n  args: {\n    file: { type: ["string", "null"], description: "Tên file CSV trong /data. Gửi null để tìm file." },\n    column: { type: ["string", "null"], description: "Tên cột cần xem các nhãn. Gửi null để xem danh sách cột." },\n  },\n  async execute(args) {\n    return run("inspect_table", args);\n  },\n};\n'
EMBEDDED_AGGREGATE_JS = 'import { run } from "../run.js";\n\nexport default {\n  description: "Đếm giá trị duy nhất trong khoảng ngày, loại cả mã có một dòng mang trạng thái bị loại.",\n  args: {\n    file: { type: "string", description: "Tên file CSV trong /data." },\n    operation: { type: "string", enum: ["count_distinct"] },\n    date_column: { type: "string", description: "Tên cột chứa ngày." },\n    date_expression: { type: "string", description: "Mốc hoặc khoảng ngày trong câu hỏi. Có thể dùng nguyên cụm ngày tương đối." },\n    as_of_date: { type: "string", description: "Ngày tham chiếu YYYY-MM-DD từ đầu vào." },\n    column: { type: "string", description: "Cột cần đếm giá trị duy nhất." },\n    status_column: { type: "string", description: "Cột trạng thái để áp dụng điều kiện loại." },\n    exclude_values: { type: "array", items: { type: "string" }, description: "Nhãn thực tế cần loại theo rulebook. Dùng [] khi không có điều kiện loại." },\n  },\n  async execute(args) {\n    return run("aggregate", args);\n  },\n};\n'
EMBEDDED_RUN_PYTHON_JS = 'import { run } from "../run.js";\n\nexport default {\n  description: "Chạy code Python chỉ đọc trên dữ liệu của shop (tính tổng, trung bình, tỷ lệ, top/xếp hạng, bảng, hoặc đọc XLSX/SQLite). Helper (explore_data, load_clean_df, resolve_date, parse_vn_date, parse_vn_number, vn_num, emit_obs, emit_final) ĐÃ CÓ SẴN trong globals (KHÔNG import). BẮT BUỘC kết thúc bằng emit_obs(data, msg) để quan sát hoặc emit_final(answer, type=..., evidence=...) để chốt kết quả — KHÔNG dùng print() suông ở cuối. Sau khi gọi emit_final thành công, BẮT BUỘC phải xuất đáp án cuối cùng trong câu trả lời (không để trống).",\n  args: {\n    code: {\n      type: "string",\n      description: "Mã Python chạy trong thư mục shop (BUSINESS_DIR). Helper có sẵn trong globals (KHÔNG import): explore_data(file, column=None, keyword=None, sheet=None, date=None) (date nhận YYYY-MM-DD, YYYY-MM hoặc cụm như hôm qua: trả các dòng của mốc đó và dòng tổng kề bên; dòng Tổng KHÔNG được cộng chung với dòng hàng), load_clean_df(file, sheet=0, dedup_order=False) (hoặc pd.read_csv/pd.read_excel), resolve_date(expr, as_of_date=AS_OF_DATE) -> {\'start_date\',\'end_date\'}, parse_vn_date(series, default_year=...), parse_vn_number(val)/vn_num(val), emit_obs(data, msg), emit_final(answer, type=..., evidence=...). KHÔNG hardcode /data; KHÔNG truyền kwargs lạ như parse_dates vào load_clean_df. Luôn kết thúc bằng emit_obs(data, msg) hoặc emit_final(answer, type=..., evidence=...). Sau khi emit_final, xuất đáp án cuối trong câu trả lời.",\n    },\n  },\n  async execute(args) {\n    return run("run_python", args);\n  },\n};\n'
EMBEDDED_EXPLORE_DATA_JS = 'import { run } from "../run.js";\n\nexport default {\n  description: "Xem dữ liệu của một file (CSV, XLSX, SQLite) ở đúng mốc thời gian của câu hỏi, trước khi tính toán. Trả các dòng hàng trong mốc, số dòng theo trạng thái, và các dòng Tổng kề bên (dòng Tổng KHÔNG được cộng chung với dòng hàng). Chỉ đọc.",\n  args: {\n    file: { type: "string", description: "Tên file trong /data." },\n    date: { type: ["string", "null"], description: "Mốc cần xem: YYYY-MM-DD, YYYY-MM, dd/mm/yyyy hoặc cụm như hôm qua, tháng này (tính theo ngày hiện tại của câu hỏi). Gửi null để xem tổng quan file." },\n    sheet: { type: ["string", "null"], description: "Tên sheet nếu file XLSX có nhiều sheet. Gửi null để dùng sheet mặc định." },\n    column: { type: ["string", "null"], description: "Cột cần xem các giá trị thực tế. Gửi null nếu không cần." },\n    keyword: { type: ["string", "null"], description: "Từ khoá cần tìm trong dữ liệu. Gửi null nếu không cần." },\n  },\n  async execute(args) {\n    return run("explore_data", args);\n  },\n};\n'
EMBEDDED_BRIDGE_PY = '"""Read-only adapter for the DataMate OpenCode tools using the pinned Khanh Linh helpers."""\n\nimport json\nimport os\nimport re\nimport sys\nimport time\nimport unicodedata\nfrom pathlib import Path\n\nimport pandas as pd\n\n_BRIDGE_DIR = Path(__file__).resolve().parent\n_CUSTOM_SRC_ROOT = os.environ.get("DATAMATE_SRC_ROOT", "").strip()\nif _CUSTOM_SRC_ROOT and Path(_CUSTOM_SRC_ROOT).exists():\n    if _CUSTOM_SRC_ROOT not in sys.path:\n        sys.path.insert(0, _CUSTOM_SRC_ROOT)\nelif (_BRIDGE_DIR / "src").is_dir():\n    if str(_BRIDGE_DIR) not in sys.path:\n        sys.path.insert(0, str(_BRIDGE_DIR))\n\nfrom src.agent.sandbox import (  # noqa: E402\n    explore_data,\n    load_clean_df,\n    resolve_date,\n    run_python_in_sandbox,\n)\nfrom src.postprocess import parse_vn_date  # noqa: E402\n\n\ndef _append_tool_trace(record: dict) -> None:\n    """Append untruncated tool execution trace when DATAMATE_TOOL_TRACE_PATH is set."""\n    trace_path = os.environ.get("DATAMATE_TOOL_TRACE_PATH", "").strip()\n    if not trace_path:\n        return\n    try:\n        target = Path(trace_path)\n        target.parent.mkdir(parents=True, exist_ok=True)\n        with target.open("a", encoding="utf-8") as fh:\n            fh.write(json.dumps(record, ensure_ascii=False) + "\\n")\n    except OSError:\n        pass\n\n\n_TABULAR_EXTENSIONS = {".csv", ".xlsx", ".xls", ".sqlite", ".db", ".sqlite3"}\n_NON_CSV_EXTENSIONS = {".xlsx", ".xls", ".sqlite", ".db", ".sqlite3"}\n\n\ndef _normalize_optional_arg(value):\n    if value is None:\n        return None\n    if isinstance(value, str) and value.strip().lower() in ("", "none", "null", "nil"):\n        return None\n    return value\n\n\ndef _try_repair_mojibake(name: str) -> str:\n    """Repair UTF-8 filenames decoded through CP437 during ZIP extraction."""\n    try:\n        repaired = name.encode("cp437").decode("utf-8")\n        if repaired:\n            return repaired\n    except (UnicodeEncodeError, UnicodeDecodeError):\n        pass\n    return name\n\n\ndef _norm_filename_key(name: str) -> str:\n    """Normalize filename across CP437 mojibake, Vietnamese accents, and space/underscore variants."""\n    repaired = _try_repair_mojibake(name).replace("đ", "d").replace("Đ", "D")\n    nfd = unicodedata.normalize("NFD", repaired.lower())\n    stripped = "".join(ch for ch in nfd if unicodedata.category(ch) != "Mn")\n    return re.sub(r"[\\s_\\-]+", "_", stripped).strip("_")\n\n\ndef _resolve_shop_file(data_root: Path, file_name: str, allowed_exts: set[str]) -> Path | None:\n    """Safely resolve a tabular file inside data_root with mojibake, accent, and stem fallbacks."""\n    if not isinstance(file_name, str) or not file_name.strip():\n        return None\n    raw_name = file_name.strip()\n    norm_slash = raw_name.replace("\\\\", "/")\n    if norm_slash.startswith("/data/"):\n        raw_name = norm_slash[len("/data/"):]\n    raw_path = Path(raw_name)\n    if ".." in raw_path.parts:\n        direct = (data_root / raw_path).resolve()\n        if direct.is_relative_to(data_root) and direct.suffix.lower() in allowed_exts and direct.is_file():\n            return direct\n        return None\n\n    direct = (data_root / raw_path).resolve()\n    if direct.is_relative_to(data_root) and direct.suffix.lower() in allowed_exts and direct.is_file():\n        return direct\n\n    is_simple_name = len(raw_path.parts) <= 1 and not raw_path.is_absolute()\n    if not is_simple_name and not (raw_path.is_absolute() and str(raw_path.parent.resolve()) == str(data_root)):\n        return None\n\n    target_name = raw_path.name\n    target_ext = Path(target_name).suffix.lower()\n    target_key = _norm_filename_key(target_name)\n    target_stem_key = _norm_filename_key(Path(target_name).stem)\n    candidates = [\n        p for p in sorted(data_root.iterdir())\n        if p.is_file() and p.suffix.lower() in allowed_exts\n    ]\n    # Pass 1: mojibake-repaired or normalized full filename match\n    for cand in candidates:\n        if _try_repair_mojibake(cand.name) == target_name or _norm_filename_key(cand.name) == target_key:\n            return cand.resolve()\n    # Pass 2: missing extension or relaxed alphanumeric stem match\n    target_alnum = "".join(ch for ch in target_stem_key if ch.isalnum())\n    for cand in candidates:\n        if target_ext and cand.suffix.lower() != target_ext:\n            continue\n        cand_stem_key = _norm_filename_key(cand.stem)\n        cand_alnum = "".join(ch for ch in cand_stem_key if ch.isalnum())\n        if cand_stem_key == target_stem_key or (target_alnum and cand_alnum == target_alnum):\n            return cand.resolve()\n    return None\n\n\ndef _looks_like_scalar_value(val: str) -> bool:\n    """Return True if a preview cell looks like a pure number or calendar date rather than a column header."""\n    s = val.strip()\n    if not s:\n        return True\n    if s.replace(".", "", 1).replace(",", "", 1).replace("-", "", 1).isdigit():\n        return True\n    return bool(re.match(r"^(?:\\d{4}-\\d{1,2}-\\d{1,2}|\\d{1,2}[/-]\\d{1,2}[/-]\\d{2,4})$", s))\n\n\ndef _detect_header_row_from_preview(raw_rows: list[list[str]]) -> tuple[int | None, list[str], list[list[str]]]:\n    """Identify the 0-based Excel header row index, column labels, and first data sample rows."""\n    max_non_empty = max((sum(1 for cell in row if cell != "") for row in raw_rows), default=0)\n    hdr_idx: int | None = None\n    for idx, row in enumerate(raw_rows):\n        non_empty = [cell for cell in row if cell != ""]\n        if len(non_empty) >= max(3, int(max_non_empty * 0.5)):\n            non_scalar = [cell for cell in non_empty if not _looks_like_scalar_value(cell)]\n            if len(non_scalar) >= len(non_empty) * 0.7:\n                hdr_idx = idx\n                break\n    if hdr_idx is None:\n        return None, [], [row for row in raw_rows if any(row)][:2]\n    hdr_row = raw_rows[hdr_idx]\n    columns = [cell if cell != "" else f"col_{i}" for i, cell in enumerate(hdr_row)]\n    sample_rows = [row for row in raw_rows[hdr_idx + 1:] if any(row)][:2]\n    return hdr_idx, columns, sample_rows\n\n\ndef _fast_inspect_xlsx(file: Path) -> dict:\n    """Quick read-only sheet and top-row header peek for XLSX without loading 100k rows."""\n    import openpyxl\n\n    wb = openpyxl.load_workbook(str(file), read_only=True, data_only=True)\n    try:\n        sheet_names = list(wb.sheetnames)\n        sheets_preview = {}\n        seen_schemas: dict[tuple[int | None, tuple[str, ...]], str] = {}\n        for sname in sheet_names[:15]:\n            ws = wb[sname]\n            raw_rows: list[list[str]] = []\n            for row in ws.iter_rows(max_row=10, values_only=True):\n                cleaned = [("" if v is None else str(v).strip()) for v in row]\n                while cleaned and cleaned[-1] == "":\n                    cleaned.pop()\n                raw_rows.append(cleaned[:25])\n            while raw_rows and not any(raw_rows[-1]):\n                raw_rows.pop()\n\n            hdr_idx, header_cols, sample_rows = _detect_header_row_from_preview(raw_rows)\n            schema_key = (hdr_idx, tuple(header_cols)) if header_cols else None\n            if schema_key is not None and schema_key in seen_schemas and len(sheets_preview) >= 2:\n                sheets_preview[sname] = {\n                    "approx_rows": ws.max_row,\n                    "approx_cols": ws.max_column,\n                    "header_row_index": hdr_idx,\n                    "same_columns_as": seen_schemas[schema_key],\n                    "sample_rows": sample_rows[:1],\n                }\n                continue\n\n            if schema_key is not None:\n                seen_schemas[schema_key] = sname\n            end_row = min(len(raw_rows), (hdr_idx + 3) if hdr_idx is not None else 6)\n            sheets_preview[sname] = {\n                "approx_rows": ws.max_row,\n                "approx_cols": ws.max_column,\n                "header_row_index": hdr_idx,\n                "columns": header_cols,\n                "first_rows": raw_rows[:end_row],\n                "sample_rows": sample_rows,\n            }\n        out = {\n            "file": file.name,\n            "type": "xlsx",\n            "sheets": sheet_names,\n            "sheets_preview": sheets_preview,\n            "hint": "Use load_clean_df(file, sheet=..., dedup_order=False) or pd.read_excel(file, sheet_name=...) — headers are auto-detected.",\n        }\n        repaired_name = _try_repair_mojibake(file.name)\n        if repaired_name != file.name:\n            out["alias"] = repaired_name\n        return out\n    finally:\n        wb.close()\n\n\ndef inspect_table(file, column=None):\n    column = _normalize_optional_arg(column)\n    if file.suffix.lower() == ".xlsx" and column is None:\n        try:\n            return _fast_inspect_xlsx(file)\n        except Exception:\n            pass\n    observation = explore_data(file, column=column, n=0)\n    if column is not None or "error" in observation:\n        return observation\n    keys = ("file", "type", "sheets", "active_sheet", "tables", "columns", "total_rows", "date_coverage")\n    return {key: observation[key] for key in keys if key in observation}\n\n\ndef count_distinct(file, date_column, date_expression, as_of_date, column, status_column, exclude_values):\n    period = resolve_date(date_expression, as_of_date=as_of_date)\n    if "start_time" in period or "end_time" in period:\n        raise ValueError("This tool supports whole days and date ranges.")\n\n    table = load_clean_df(file, id_col=column, date_col=date_column, dedup_order=False)\n    statuses = table[status_column].astype("string").str.strip()\n    observed = set(statuses.dropna().unique())\n    exclusions = {value.strip() for value in exclude_values}\n    if unknown := exclusions - observed:\n        raise ValueError(f"Unknown status values {sorted(unknown)}. Observed statuses: {sorted(observed)}")\n\n    dates = parse_vn_date(table[date_column], default_year=table.attrs.get("default_year"))\n    start = pd.Timestamp(period["start_date"])\n    stop = pd.Timestamp(period["end_date"]) + pd.Timedelta(days=1)\n    in_period = dates.ge(start) & dates.lt(stop)\n    order_ids = table[column].astype("string").str.strip().replace("", pd.NA)\n    all_ids = set(order_ids[in_period].dropna())\n    excluded_ids = set(order_ids[in_period & statuses.isin(exclusions)].dropna())\n    return {\n        "type": "number",\n        "answer": len(all_ids - excluded_ids),\n        "start_date": period["start_date"],\n        "end_date": period["end_date"],\n        "matched_rows": int(in_period.sum()),\n        "distinct_before_exclusion": len(all_ids),\n        "excluded_distinct": len(excluded_ids),\n    }\n\n\n_BRIDGE_RUNTIME_PATCH = r\'\'\'\n_BRIDGE_ALLOWED_EXTS = (\'.csv\', \'.xlsx\', \'.xls\', \'.sqlite\', \'.db\', \'.sqlite3\')\n\ndef _bridge_repair_mojibake(name):\n    try:\n        repaired = str(name).encode(\'cp437\').decode(\'utf-8\')\n        if repaired:\n            return repaired\n    except Exception:\n        pass\n    return str(name)\n\ndef _bridge_norm_file_key(name):\n    repaired = _bridge_repair_mojibake(name).replace(\'đ\', \'d\').replace(\'Đ\', \'D\')\n    nfd = \'\'.join(c for c in unicodedata.normalize(\'NFD\', repaired.lower()) if unicodedata.category(c) != \'Mn\')\n    return _re.sub(r\'[\\s_\\-]+\', \'_\', nfd).strip(\'_\')\n\ndef _smart_resolve_file(path_or_str, business_dir=None):\n    if not isinstance(path_or_str, (str, Path)):\n        return path_or_str\n    p = str(path_or_str)\n    if os.path.exists(p):\n        return p\n    bdir = str(business_dir or globals().get(\'BUSINESS_DIR\', \'\') or os.environ.get(\'BUSINESS_DIR\', \'\'))\n    if not bdir or not os.path.isdir(bdir):\n        return p\n    norm_p = p.replace(\'\\\\\', \'/\')\n    if norm_p.startswith(\'/data/\'):\n        norm_p = norm_p[len(\'/data/\'):]\n    base = os.path.basename(norm_p)\n    if not base:\n        return p\n    direct = os.path.join(bdir, base)\n    if os.path.exists(direct):\n        return direct\n    ext = os.path.splitext(base)[1].lower()\n    target_key = _bridge_norm_file_key(base)\n    target_stem_key = _bridge_norm_file_key(os.path.splitext(base)[0])\n    target_alnum = \'\'.join(c for c in target_stem_key if c.isalnum())\n    cands = sorted(os.listdir(bdir))\n    for cand in cands:\n        if _bridge_repair_mojibake(cand) == base or _bridge_norm_file_key(cand) == target_key:\n            return os.path.join(bdir, cand)\n    for cand in cands:\n        c_stem, c_ext = os.path.splitext(cand)\n        c_ext_low = c_ext.lower()\n        if ext and c_ext_low != ext:\n            continue\n        if not ext and c_ext_low not in _BRIDGE_ALLOWED_EXTS:\n            continue\n        c_stem_key = _bridge_norm_file_key(c_stem)\n        c_alnum = \'\'.join(c for c in c_stem_key if c.isalnum())\n        if c_stem_key == target_stem_key or (target_alnum and (target_alnum == c_alnum or target_alnum in c_alnum or c_alnum in target_alnum)):\n            return os.path.join(bdir, cand)\n    return p\n\ndef _bridge_detect_best_header(df):\n    if not isinstance(df, pd.DataFrame) or len(df) == 0:\n        return df\n    cols = [str(c) for c in df.columns]\n    unnamed_cnt = sum(1 for c in cols if c.startswith(\'Unnamed:\') or c == \'nan\' or not c.strip())\n    if unnamed_cnt / max(1, len(cols)) < 0.4:\n        return df\n    for r_idx in range(min(10, len(df))):\n        row_vals = df.iloc[r_idx].values\n        str_vals = [str(v).strip() for v in row_vals if pd.notna(v) and str(v).strip() != \'\' and str(v) != \'nan\']\n        if len(str_vals) >= max(3, len(cols) * 0.5):\n            non_num = [v for v in str_vals if not v.replace(\'.\', \'\', 1).replace(\',\', \'\', 1).replace(\'-\', \'\', 1).isdigit()]\n            if len(non_num) >= len(str_vals) * 0.7:\n                banner_parts = [c for c in cols if c and not c.startswith(\'Unnamed:\') and c.lower() != \'nan\']\n                for b_idx in range(r_idx):\n                    for v in df.iloc[b_idx].values:\n                        sv = str(v).strip()\n                        if pd.notna(v) and sv and sv.lower() != \'nan\':\n                            banner_parts.append(sv)\n                new_cols = [str(v).strip() if pd.notna(v) and str(v).strip() != \'\' else f\'col_{i}\' for i, v in enumerate(row_vals)]\n                new_df = df.iloc[r_idx + 1:].copy()\n                new_df.columns = new_cols\n                new_df.reset_index(drop=True, inplace=True)\n                if hasattr(new_df, \'attrs\') and isinstance(new_df.attrs, dict) and banner_parts:\n                    new_df.attrs[\'_banner_text\'] = \' \'.join(banner_parts)\n                return new_df\n    return df\n\n_detect_best_header = _bridge_detect_best_header\n\nclass _BridgeSheetDict(dict):\n    @property\n    def sheet_names(self):\n        return list(self.keys())\n    @property\n    def sheetnames(self):\n        return list(self.keys())\n\n_BRIDGE_EXCEL_SHEETS_CACHE = {}\n\ndef _bridge_get_excel_sheets(path_str):\n    key = str(path_str)\n    if key not in _BRIDGE_EXCEL_SHEETS_CACHE:\n        try:\n            with _orig_excel_file(key) as _xf:\n                _BRIDGE_EXCEL_SHEETS_CACHE[key] = list(_xf.sheet_names)\n        except Exception:\n            _BRIDGE_EXCEL_SHEETS_CACHE[key] = []\n    return list(_BRIDGE_EXCEL_SHEETS_CACHE[key])\n\nif not hasattr(pd.DataFrame, \'sheetnames\'):\n    @property\n    def _bridge_df_sheetnames(self):\n        if isinstance(self.columns, pd.MultiIndex) or \'sheetnames\' not in self.columns:\n            return list(self.attrs.get(\'_excel_sheet_names\', []))\n        return self[\'sheetnames\']\n    pd.DataFrame.sheetnames = _bridge_df_sheetnames\n\nif not hasattr(pd.DataFrame, \'sheet_names\'):\n    @property\n    def _bridge_df_sheet_names(self):\n        if isinstance(self.columns, pd.MultiIndex) or \'sheet_names\' not in self.columns:\n            return list(self.attrs.get(\'_excel_sheet_names\', []))\n        return self[\'sheet_names\']\n    pd.DataFrame.sheet_names = _bridge_df_sheet_names\n\ndef _bridge_smart_read_csv(filepath_or_buffer, *args, **kwargs):\n    if isinstance(filepath_or_buffer, (str, Path)):\n        filepath_or_buffer = _smart_resolve_file(filepath_or_buffer)\n    res = _orig_read_csv(filepath_or_buffer, *args, **kwargs)\n    if isinstance(res, pd.DataFrame):\n        res = _bridge_detect_best_header(res)\n    return res\n\npd.read_csv = _bridge_smart_read_csv\n\ndef _bridge_smart_read_excel(io, *args, **kwargs):\n    unmerge_flag = kwargs.pop(\'unmerge\', None)\n    if isinstance(io, (str, Path)):\n        io_str = str(io)\n        m_qs = _re.search(r\'[&?#]sheet=([^&?#]+)$\', io_str)\n        if m_qs:\n            io_str = io_str[:m_qs.start()]\n            kwargs.setdefault(\'sheet_name\', m_qs.group(1))\n        io = _smart_resolve_file(io_str)\n    sheet = kwargs.get(\'sheet_name\')\n    def _post_excel(obj, src_path=io):\n        if isinstance(obj, pd.DataFrame):\n            obj = _bridge_detect_best_header(obj)\n            if unmerge_flag or unmerge_flag is None:\n                obj = unmerge_and_ffill(obj)\n            if isinstance(src_path, (str, Path)) and os.path.exists(str(src_path)):\n                obj.attrs[\'_excel_sheet_names\'] = _bridge_get_excel_sheets(src_path)\n            return obj\n        if isinstance(obj, dict):\n            snames = list(obj.keys())\n            out = {}\n            for k, v in obj.items():\n                nv = _post_excel(v, src_path=src_path)\n                if isinstance(nv, pd.DataFrame):\n                    nv.attrs[\'_excel_sheet_names\'] = snames\n                out[k] = nv\n            return _BridgeSheetDict(out)\n        return obj\n    try:\n        res = _orig_read_excel(io, *args, **kwargs)\n        return _post_excel(res, src_path=io)\n    except (ValueError, KeyError, FileNotFoundError):\n        if sheet and isinstance(sheet, str):\n            bdir = str(globals().get(\'BUSINESS_DIR\', \'\') or os.environ.get(\'BUSINESS_DIR\', \'\'))\n            if bdir and os.path.isdir(bdir):\n                cur_base = os.path.basename(str(io))\n                for fname in sorted(os.listdir(bdir)):\n                    if fname.lower().endswith((\'.xlsx\', \'.xls\')) and fname != cur_base:\n                        other_path = os.path.join(bdir, fname)\n                        try:\n                            other_xls = pd.ExcelFile(other_path)\n                            if sheet in other_xls.sheet_names:\n                                res = _orig_read_excel(other_path, *args, **kwargs)\n                                return _post_excel(res, src_path=other_path)\n                        except Exception:\n                            pass\n        raise\n\npd.read_excel = _bridge_smart_read_excel\n\ntry:\n    from pandas.core.strings.accessor import StringMethods as _BridgeStrMethods\n    _bridge_orig_str_contains = _BridgeStrMethods.contains\n\n    def _bridge_str_contains(self, pat, *args, **kwargs):\n        res = _bridge_orig_str_contains(self, pat, *args, **kwargs)\n        try:\n            if isinstance(res, pd.Series) and not bool(res.dropna().any()) and isinstance(pat, str) and pat.strip():\n                clean_pat = pat.replace(r\'\\(\', \'(\').replace(r\'\\)\', \')\').replace(r\'\\[\', \'[\').replace(r\'\\]\', \']\')\n                retry_kw = {\'case\': False, \'regex\': False}\n                if \'na\' in kwargs:\n                    retry_kw[\'na\'] = kwargs[\'na\']\n                elif len(args) >= 3:\n                    retry_kw[\'na\'] = args[2]\n                res2 = _bridge_orig_str_contains(self, clean_pat, **retry_kw)\n                if isinstance(res2, pd.Series) and bool(res2.dropna().any()):\n                    return res2\n        except Exception:\n            pass\n        return res\n\n    _BridgeStrMethods.contains = _bridge_str_contains\nexcept Exception:\n    pass\n\nimport atexit as _bridge_atexit\nimport builtins as _bridge_builtins\nimport types as _bridge_types\n\ntry:\n    _bridge_orig_df_getitem = pd.DataFrame.__getitem__\n    def _bridge_df_getitem(self, key):\n        try:\n            return _bridge_orig_df_getitem(self, key)\n        except (KeyError, TypeError):\n            if isinstance(key, str) and hasattr(self, \'columns\'):\n                key_strip = key.strip()\n                cols = [str(c) for c in self.columns]\n                for orig_col, col_str in zip(self.columns, cols):\n                    if key_strip.lower() == col_str.strip().lower():\n                        return _bridge_orig_df_getitem(self, orig_col)\n                def _norm_k(s):\n                    s_repaired = _bridge_repair_mojibake(s).replace(\'đ\', \'d\').replace(\'Đ\', \'D\')\n                    nfd = \'\'.join(c for c in unicodedata.normalize(\'NFD\', s_repaired.lower()) if unicodedata.category(c) != \'Mn\')\n                    return _re.sub(r\'[\\s_\\-]+\', \'\', nfd)\n                target_norm = _norm_k(key_strip)\n                for orig_col, col_str in zip(self.columns, cols):\n                    if _norm_k(col_str) == target_norm:\n                        return _bridge_orig_df_getitem(self, orig_col)\n                if target_norm in (\'tanggal\', \'date\', \'ngay\', \'time\', \'thoigian\'):\n                    for orig_col, col_str in zip(self.columns, cols):\n                        c_norm = _norm_k(col_str)\n                        if any(w in c_norm for w in (\'ngay\', \'date\', \'thoigian\', \'lap\')):\n                            return _bridge_orig_df_getitem(self, orig_col)\n            raise\n    pd.DataFrame.__getitem__ = _bridge_df_getitem\nexcept Exception:\n    pass\n\ntry:\n    _bridge_orig_to_numeric = pd.to_numeric\n    def _bridge_to_numeric(arg, *args, **kwargs):\n        try:\n            res = _bridge_orig_to_numeric(arg, *args, **kwargs)\n            if isinstance(res, pd.Series):\n                if len(res) > 0 and res.isna().sum() > (arg.isna().sum() if hasattr(arg, \'isna\') else 0):\n                    try:\n                        vn_res = arg.apply(parse_vn_number)\n                        if vn_res.notna().sum() > res.notna().sum():\n                            return vn_res\n                    except Exception:\n                        pass\n            elif (res is None or (isinstance(res, float) and pd.isna(res))) and not pd.isna(arg):\n                try:\n                    vn_val = parse_vn_number(arg)\n                    if not pd.isna(vn_val):\n                        return vn_val\n                except Exception:\n                    pass\n            return res\n        except Exception:\n            try:\n                if isinstance(arg, pd.Series):\n                    return arg.apply(parse_vn_number)\n                return parse_vn_number(arg)\n            except Exception:\n                raise\n    pd.to_numeric = _bridge_to_numeric\nexcept Exception:\n    pass\n\nnumber = \'number\'\nstring = \'string\'\nboolean = \'boolean\'\ntable = \'table\'\nvn_num = parse_vn_number\n\ntry:\n    import src.agent.sandbox as _sb_mod\n    _sb_mod._smart_resolve_file = _smart_resolve_file\n    _sb_mod._detect_best_header = _bridge_detect_best_header\n    _sb_mod._READ_EXCEL = _bridge_smart_read_excel\n    _sb_mod._READ_CSV = _bridge_smart_read_csv\nexcept Exception:\n    pass\n\n_BRIDGE_STATE = {\n    \'emitted\': False,\n    \'printed\': False,\n    \'printed_after_emit\': False,\n    \'last_emitted_json_line\': None,\n    \'last_explore\': None,\n    \'had_exception\': False,\n    \'empty_final_called\': False,\n    \'stdout_parts\': [],\n}\n\n_bridge_orig_excepthook = sys.excepthook\ndef _bridge_excepthook(exc_type, exc_val, exc_tb):\n    _BRIDGE_STATE[\'had_exception\'] = True\n    return _bridge_orig_excepthook(exc_type, exc_val, exc_tb)\nsys.excepthook = _bridge_excepthook\n\n_bridge_orig_print = _bridge_builtins.print\ndef _bridge_tracked_print(*args, **kwargs):\n    _BRIDGE_STATE[\'printed\'] = True\n    if _BRIDGE_STATE[\'emitted\']:\n        _BRIDGE_STATE[\'printed_after_emit\'] = True\n    try:\n        sep = kwargs.get(\'sep\', \' \')\n        end = kwargs.get(\'end\', \'\\n\')\n        text = sep.join(str(a) for a in args) + end\n        _BRIDGE_STATE[\'stdout_parts\'].append(text)\n    except Exception:\n        pass\n    return _bridge_orig_print(*args, **kwargs)\n_bridge_builtins.print = _bridge_tracked_print\n\n_bridge_orig_explore_data = explore_data\ndef _bridge_explore_data(*args, **kwargs):\n    res = _bridge_orig_explore_data(*args, **kwargs)\n    _BRIDGE_STATE[\'last_explore\'] = res\n    return res\nexplore_data = _bridge_explore_data\n\n_bridge_orig_emit_obs = emit_obs\ndef _bridge_emit_obs(*args, **kwargs):\n    res = _bridge_orig_emit_obs(*args, **kwargs)\n    _BRIDGE_STATE[\'emitted\'] = True\n    _BRIDGE_STATE[\'printed_after_emit\'] = False\n    try:\n        _BRIDGE_STATE[\'last_emitted_json_line\'] = json.dumps(res, ensure_ascii=False)\n    except Exception:\n        pass\n    return res\nemit_obs = _bridge_emit_obs\n\n_TYPE_SYNONYMS = {\n    \'int\': \'number\', \'float\': \'number\', \'integer\': \'number\', \'num\': \'number\',\n    \'str\': \'string\', \'text\': \'string\',\n    \'bool\': \'boolean\',\n    \'array\': \'list\', \'sequence\': \'list\',\n    \'dict\': \'table\', \'ranking\': \'table\',\n}\n\n_bridge_orig_emit_final = emit_final\n_BRIDGE_EMIT_FINAL_ALLOWED_KW = (\'answer\', \'type\', \'evidence\', \'check\', \'reason\')\ndef _bridge_emit_final(*args, **kwargs):\n    if not args and \'answer\' not in kwargs:\n        _BRIDGE_STATE[\'empty_final_called\'] = True\n        return None\n    if args and \'answer\' in kwargs:\n        kw_ans = kwargs.pop(\'answer\')\n        if args[0] is None and kw_ans is not None:\n            args = (kw_ans,) + args[1:]\n\n    # Map type synonyms\n    if \'type\' in kwargs:\n        t_val = kwargs[\'type\']\n        if t_val is int or t_val is float:\n            kwargs[\'type\'] = \'number\'\n        elif t_val is str:\n            kwargs[\'type\'] = \'string\'\n        elif t_val is bool:\n            kwargs[\'type\'] = \'boolean\'\n        elif t_val is _bridge_builtins.list or t_val is list:\n            kwargs[\'type\'] = \'list\'\n        elif t_val is dict:\n            kwargs[\'type\'] = \'table\'\n        elif isinstance(t_val, str):\n            t_clean = t_val.strip().lower()\n            kwargs[\'type\'] = _TYPE_SYNONYMS.get(t_clean, t_clean)\n        elif hasattr(t_val, \'__name__\'):\n            t_name = t_val.__name__.lower()\n            kwargs[\'type\'] = _TYPE_SYNONYMS.get(t_name, t_name)\n    elif len(args) >= 2:\n        t_val = args[1]\n        mapped_t = None\n        if t_val is int or t_val is float:\n            mapped_t = \'number\'\n        elif t_val is str:\n            mapped_t = \'string\'\n        elif t_val is bool:\n            mapped_t = \'boolean\'\n        elif t_val is _bridge_builtins.list or t_val is list:\n            mapped_t = \'list\'\n        elif t_val is dict:\n            mapped_t = \'table\'\n        elif isinstance(t_val, str):\n            t_clean = t_val.strip().lower()\n            mapped_t = _TYPE_SYNONYMS.get(t_clean, t_clean)\n        elif hasattr(t_val, \'__name__\'):\n            t_name = t_val.__name__.lower()\n            mapped_t = _TYPE_SYNONYMS.get(t_name, t_name)\n        if mapped_t is not None:\n            args = (args[0], mapped_t) + args[2:]\n\n    # Evidence aliases\n    if not kwargs.get(\'evidence\') and len(args) < 3:\n        for _alt_k in (\'msg\', \'message\', \'detail\', \'note\', \'explanation\'):\n            _alt_v = kwargs.get(_alt_k)\n            if isinstance(_alt_v, str) and _alt_v.strip():\n                kwargs[\'evidence\'] = _alt_v.strip()\n                break\n\n    # Default evidence if empty or missing\n    if len(args) < 3:\n        if not kwargs.get(\'evidence\') or not str(kwargs.get(\'evidence\', \'\')).strip():\n            kwargs[\'evidence\'] = \'Computed via sandbox\'\n    elif len(args) >= 3 and (args[2] is None or not str(args[2]).strip()):\n        args = (args[0], args[1], \'Computed via sandbox\') + args[3:]\n\n    for _k in list(kwargs.keys()):\n        if _k not in _BRIDGE_EMIT_FINAL_ALLOWED_KW:\n            kwargs.pop(_k, None)\n\n    _BRIDGE_STATE[\'emitted\'] = True\n    _BRIDGE_STATE[\'printed_after_emit\'] = False\n    res = _bridge_orig_emit_final(*args, **kwargs)\n    try:\n        _BRIDGE_STATE[\'last_emitted_json_line\'] = json.dumps(res, ensure_ascii=False)\n    except Exception:\n        pass\n    return res\nemit_final = _bridge_emit_final\n\ndef _bridge_auto_emit_on_exit():\n    if _BRIDGE_STATE[\'had_exception\']:\n        return\n    if _BRIDGE_STATE[\'emitted\']:\n        if _BRIDGE_STATE[\'printed_after_emit\'] and _BRIDGE_STATE[\'last_emitted_json_line\']:\n            _bridge_orig_print(_BRIDGE_STATE[\'last_emitted_json_line\'])\n        return\n    if _BRIDGE_STATE[\'empty_final_called\']:\n        obs_data = (\n            _BRIDGE_STATE[\'last_explore\']\n            if _BRIDGE_STATE[\'last_explore\'] is not None\n            else {\'note\': \'Exploratory stdout captured; use emit_final(answer, type=..., evidence=...) to submit final answer.\'}\n        )\n        _bridge_orig_emit_obs(\n            obs_data,\n            \'emit_final() without answer converted to observation. To submit final result, call emit_final(answer, type=..., evidence=...).\',\n        )\n    elif _BRIDGE_STATE[\'last_explore\'] is not None and not _BRIDGE_STATE[\'printed\']:\n        _bridge_orig_emit_obs(_BRIDGE_STATE[\'last_explore\'], \'explore_data result\')\n    elif os.environ.get(\'DATAMATE_AUTO_WRAP_STDOUT\', \'0\') == \'1\':\n        captured = "".join(_BRIDGE_STATE[\'stdout_parts\']).strip()\n        if not captured and _BRIDGE_STATE[\'last_explore\'] is not None:\n            captured = _BRIDGE_STATE[\'last_explore\']\n        elif not captured:\n            captured = "Execution finished successfully with no output."\n        _bridge_orig_emit_obs(captured, "Captured output")\n\n_bridge_atexit.register(_bridge_auto_emit_on_exit)\n\nfor _mod_name in (\'ShopExplorer\', \'shop_explorer\', \'openclaw_kernels_helpers\', \'datamate_helpers\', \'parse_vn_number_helper\', \'helpers\', \'datamate\', \'sandbox_helpers\'):\n    _se_mod = _bridge_types.ModuleType(_mod_name)\n    _se_mod.explore_data = explore_data\n    _se_mod.load_clean_df = load_clean_df\n    _se_mod.fuzzy_find = fuzzy_find\n    _se_mod.resolve_date = resolve_date\n    _se_mod.emit_obs = emit_obs\n    _se_mod.emit_final = emit_final\n    _se_mod.parse_vn_number = parse_vn_number\n    _se_mod.vn_num = vn_num\n    _se_mod.parse_vn_date = parse_vn_date\n    _se_mod.vn_norm = vn_norm\n    sys.modules[_mod_name] = _se_mod\n\ntry:\n    import openpyxl as _openpyxl\n    _bridge_orig_load_workbook = _openpyxl.load_workbook\n\n    class _BridgeWorkbookProxy:\n        def __init__(self, wb, resolved_path, wb_args, wb_kwargs):\n            self._wb = wb\n            self._resolved_path = str(resolved_path)\n            self._wb_args = tuple(wb_args)\n            self._wb_kwargs = dict(wb_kwargs)\n            self._extra_wbs = []\n\n        def __getattr__(self, name):\n            return getattr(self._wb, name)\n\n        def _find_sheet_in_other_wb(self, key):\n            if not isinstance(key, str):\n                return None\n            bdir = str(globals().get(\'BUSINESS_DIR\', \'\') or os.environ.get(\'BUSINESS_DIR\', \'\'))\n            if not bdir or not os.path.isdir(bdir):\n                return None\n            cur_base = os.path.basename(self._resolved_path)\n            for fname in sorted(os.listdir(bdir)):\n                if fname.lower().endswith((\'.xlsx\', \'.xlsm\')) and fname != cur_base:\n                    other_p = os.path.join(bdir, fname)\n                    try:\n                        owb = _bridge_orig_load_workbook(other_p, *self._wb_args, **self._wb_kwargs)\n                        if key in owb.sheetnames:\n                            self._extra_wbs.append(owb)\n                            return owb\n                        owb.close()\n                    except Exception:\n                        pass\n            return None\n\n        def __contains__(self, key):\n            return key in self._wb.sheetnames or self._find_sheet_in_other_wb(key) is not None\n\n        def __getitem__(self, key):\n            if key in self._wb.sheetnames:\n                return self._wb[key]\n            other_wb = self._find_sheet_in_other_wb(key)\n            if other_wb is not None:\n                return other_wb[key]\n            return self._wb[key]\n\n        def __iter__(self):\n            return iter(self._wb)\n\n        def close(self):\n            for extra in self._extra_wbs:\n                try:\n                    extra.close()\n                except Exception:\n                    pass\n            self._extra_wbs.clear()\n            return self._wb.close()\n\n        def __enter__(self):\n            return self\n\n        def __exit__(self, exc_type, exc_val, exc_tb):\n            self.close()\n            return False\n\n    def _bridge_load_workbook(filename, *args, **kwargs):\n        if isinstance(filename, (str, Path)):\n            filename = _smart_resolve_file(filename)\n        wb = _bridge_orig_load_workbook(filename, *args, **kwargs)\n        return _BridgeWorkbookProxy(wb, filename, args, kwargs)\n\n    _openpyxl.load_workbook = _bridge_load_workbook\nexcept Exception:\n    pass\n\'\'\'\n\n\ndef _cap_bridge_text(text: str | None, max_chars: int = 3500) -> str:\n    """Keep head and tail of long stdout/stderr/error text within bounded budget."""\n    s = text or ""\n    if len(s) <= max_chars:\n        return s\n    if max_chars <= 120:\n        return s[:max_chars]\n    head = max(100, int(max_chars * 0.55) - 40)\n    tail = max(100, max_chars - head - 80)\n    dropped = len(s) - head - tail\n    return f"{s[:head]}\\n... [truncated {dropped} chars; filter/aggregate before printing] ...\\n{s[-tail:]}"\n\n\ndef _bound_run_python_payload(payload: dict, raw_stdout: str | None, raw_stderr: str | None, max_json_chars: int = 3500) -> dict:\n    """Ensure serialized run_python JSON stays below OpenCode\'s inline truncation threshold."""\n    serialized = json.dumps(payload, ensure_ascii=False)\n    if len(serialized) <= max_json_chars:\n        return payload\n    over = len(serialized) - (max_json_chars - 200)\n    cur_stdout = payload.get("stdout", "")\n    if cur_stdout:\n        new_budget = max(300, len(cur_stdout) - over)\n        payload["stdout"] = _cap_bridge_text(raw_stdout, new_budget)\n    serialized = json.dumps(payload, ensure_ascii=False)\n    if len(serialized) > max_json_chars and payload.get("stderr"):\n        payload["stderr"] = _cap_bridge_text(raw_stderr, 400)\n        serialized = json.dumps(payload, ensure_ascii=False)\n    if len(serialized) > max_json_chars and payload.get("error"):\n        payload["error"] = _cap_bridge_text(payload["error"], 1200)\n    return payload\n\n\ndef run_python_tool(data_root: Path, code: str) -> dict:\n    """Execute model-generated Python inside the pinned bounded sandbox."""\n    if not isinstance(code, str) or not code.strip():\n        raise ValueError("Parameter \'code\' must be a non-empty Python code string.")\n    if not data_root.is_dir():\n        raise ValueError(f"Shop data root does not exist: {data_root}")\n\n    work_dir = os.environ.get("DATAMATE_WORK_DIR", "/tmp/datamate_work")\n    timeout_sec = int(os.environ.get("DATAMATE_TIMEOUT_SEC", "15"))\n    res = run_python_in_sandbox(\n        code=f"{_BRIDGE_RUNTIME_PATCH}\\n{code}",\n        business_dir=str(data_root),\n        calendar_path=os.environ.get("DATAMATE_CALENDAR_PATH", ""),\n        work_dir=work_dir,\n        timeout_sec=timeout_sec,\n        as_of_date=os.environ.get("DATAMATE_AS_OF_DATE", ""),\n        rulebook_path=os.environ.get("DATAMATE_RULEBOOK_PATH", ""),\n    )\n    if res.success:\n        payload = {\n            "success": True,\n            "kind": res.kind,\n            "parsed_output": res.parsed_output,\n            "stdout": _cap_bridge_text(res.stdout, 2500),\n            "stderr": _cap_bridge_text(res.stderr, 800),\n            "returncode": res.returncode,\n            "timeout": res.timeout,\n        }\n        return _bound_run_python_payload(payload, res.stdout, res.stderr, max_json_chars=3500)\n    err_msg = res.error_message or ""\n    if res.error_kind == "protocol_violation":\n        reminder = "(Protocol reminder: do NOT end with plain print(); end with emit_obs(data, msg) or emit_final(answer, type=..., evidence=...).)"\n        err_msg = f"{reminder}\\n{err_msg}" if err_msg else reminder\n        stdout_tail = (res.stdout or "").strip()[-2000:]\n        if stdout_tail and stdout_tail not in err_msg:\n            err_msg += f"\\nCaptured stdout:\\n{stdout_tail}"\n    elif res.error_kind == "timeout":\n        err_msg += " (Timed out; for large XLSX files, inspect sheets first and read only needed sheets/columns with openpyxl read_only=True or usecols.)"\n    payload = {\n        "success": False,\n        "kind": res.kind,\n        "error_kind": res.error_kind,\n        "error": _cap_bridge_text(err_msg, 2500),\n        "stdout": _cap_bridge_text(res.stdout, 2000),\n        "stderr": _cap_bridge_text(res.stderr, 800),\n        "returncode": res.returncode,\n        "timeout": res.timeout,\n    }\n    return _bound_run_python_payload(payload, res.stdout, res.stderr, max_json_chars=3500)\n\n\ndef _list_shop_files_summary(data_root: Path) -> dict:\n    """List CSV files and deduplicated non-CSV files with sheet names and repaired aliases."""\n    csv_files = sorted(path.name for path in data_root.glob("*.csv") if path.is_file())\n    raw_other_paths = sorted(\n        path for path in data_root.iterdir()\n        if path.is_file() and path.suffix.lower() in _NON_CSV_EXTENSIONS\n    )\n    dedup_map: dict[str, Path] = {}\n    for p in raw_other_paths:\n        nkey = _norm_filename_key(p.name)\n        existing = dedup_map.get(nkey)\n        if existing is None:\n            dedup_map[nkey] = p\n        elif _try_repair_mojibake(existing.name) != existing.name and _try_repair_mojibake(p.name) == p.name:\n            # Prefer clean UTF-8 filename over CP437 mojibake duplicate\n            dedup_map[nkey] = p\n    other_paths = sorted(dedup_map.values(), key=lambda x: x.name)\n\n    out: dict = {"files": csv_files}\n    if other_paths:\n        other_items = []\n        for p in other_paths:\n            item = {"file": p.name, "size_kb": round(p.stat().st_size / 1024, 1)}\n            repaired_name = _try_repair_mojibake(p.name)\n            if repaired_name != p.name:\n                item["alias"] = repaired_name\n            if p.suffix.lower() == ".xlsx":\n                try:\n                    import openpyxl\n\n                    wb = openpyxl.load_workbook(str(p), read_only=True, data_only=True)\n                    try:\n                        item["sheets"] = list(wb.sheetnames)\n                    finally:\n                        wb.close()\n                except Exception:\n                    pass\n            other_items.append(item)\n        out["other_files"] = other_items\n        out["hint"] = (\n            "Shop has non-CSV files. Use inspect_table(file=\'<name>\') for sheets/headers "\n            "or run_python with explore_data(\'<name>\') / load_clean_df(\'<name>\') / openpyxl / sqlite3."\n        )\n    return out\n\n\ndef main():\n    action = sys.argv[1]\n    raw_args = json.loads(sys.argv[2])\n    args = dict(raw_args)\n    raw_data_root = os.environ.get("DATAMATE_DATA_ROOT", "/data")\n    data_root = Path(raw_data_root).resolve()\n    data_root_label = "/data" if raw_data_root == "/data" else str(data_root)\n    os.environ["BUSINESS_DIR"] = str(data_root)\n\n    if action == "run_python":\n        code = args.pop("code", None)\n        if args:\n            raise ValueError(f"Unexpected arguments for run_python: {sorted(args)}")\n        return run_python_tool(data_root, code), raw_args, data_root\n\n    if action == "explore_data":\n        # Same sandbox as run_python: as_of_date comes from the harness env, never from the model.\n        unknown = sorted(set(args) - {"file", "date", "sheet", "column", "keyword"})\n        if unknown:\n            raise ValueError(f"Unexpected arguments for explore_data: {unknown}")\n        kwargs = {k: _normalize_optional_arg(v) for k, v in args.items()}\n        kwargs = {k: v for k, v in kwargs.items() if v is not None}\n        if not isinstance(kwargs.get("file"), str):\n            raise ValueError("Parameter \'file\' must be a file name under the shop data folder.")\n        code = f"emit_obs(explore_data(**{kwargs!r}, as_of_date=AS_OF_DATE), \'explore_data\')"\n        return run_python_tool(data_root, code), raw_args, data_root\n\n    file_name = _normalize_optional_arg(args.pop("file", None))\n    if "column" in args:\n        args["column"] = _normalize_optional_arg(args["column"])\n    if action == "inspect_table" and isinstance(file_name, str):\n        readme_norm = file_name.strip().replace("\\\\", "/")\n        if readme_norm.startswith("/data/"):\n            readme_norm = readme_norm[len("/data/"):]\n        if readme_norm.lower() in ("readme.md", "readme"):\n            out = _list_shop_files_summary(data_root)\n            readme_path = next(\n                (p for p in data_root.iterdir() if p.is_file() and p.name.lower() == "readme.md"),\n                None,\n            )\n            if readme_path is not None:\n                out["readme"] = readme_path.read_text(encoding="utf-8", errors="replace")\n            return out, raw_args, data_root\n    if file_name is None and action == "inspect_table":\n        csv_files = sorted(path.name for path in data_root.glob("*.csv") if path.is_file())\n        other_paths = sorted(\n            path for path in data_root.iterdir()\n            if path.is_file() and path.suffix.lower() in _NON_CSV_EXTENSIONS\n        )\n        if len(csv_files) == 1 and not other_paths:\n            file_name = csv_files[0]\n        else:\n            return _list_shop_files_summary(data_root), raw_args, data_root\n    if not isinstance(file_name, str) or not file_name.strip():\n        raise ValueError(f"Use a tabular file under {data_root_label}.")\n    allowed_exts = _TABULAR_EXTENSIONS if action == "inspect_table" else {".csv"}\n    file = _resolve_shop_file(data_root, file_name, allowed_exts)\n    if file is None:\n        if action == "inspect_table":\n            raise ValueError(f"Use a CSV/XLSX/SQLite file under {data_root_label}.")\n        raise ValueError(f"Use a CSV file under {data_root_label}.")\n    if action == "inspect_table":\n        return inspect_table(file, **args), raw_args, data_root\n    if args.pop("operation") != "count_distinct":\n        raise ValueError("This pilot supports count_distinct.")\n    return count_distinct(file, **args), raw_args, data_root\n\n\nif __name__ == "__main__":\n    action_name = sys.argv[1] if len(sys.argv) > 1 else ""\n    parsed_args = None\n    root_str = os.environ.get("DATAMATE_DATA_ROOT", "/data")\n    try:\n        parsed_args = json.loads(sys.argv[2]) if len(sys.argv) > 2 else {}\n        result, raw_args_out, resolved_root = main()\n        parsed_args = raw_args_out\n        root_str = str(resolved_root)\n    except (KeyError, TypeError, ValueError, OSError, json.JSONDecodeError) as error:\n        result = {"error": str(error)}\n    _append_tool_trace({\n        "timestamp": time.time(),\n        "action": action_name,\n        "data_root": root_str,\n        "args": parsed_args,\n        "result": result,\n    })\n    print(json.dumps(result, ensure_ascii=False))\n'

# Luôn ghi runtime JS/bridge đồng bộ theo bản nhúng mới nhất trong notebook (khớp 100% với tmp/opencode-structured)
for rel_file, canonical_text in [
    ("run.js", EMBEDDED_RUN_JS),
    ("bridge.py", EMBEDDED_BRIDGE_PY),
    ("tools/inspect_table.js", EMBEDDED_INSPECT_TABLE_JS),
    ("tools/aggregate.js", EMBEDDED_AGGREGATE_JS),
    ("tools/run_python.js", EMBEDDED_RUN_PYTHON_JS),
    ("tools/explore_data.js", EMBEDDED_EXPLORE_DATA_JS),
]:
    dst_f = OPENCODE_RUNTIME_DIR / rel_file
    dst_f.parent.mkdir(parents=True, exist_ok=True)
    dst_f.write_text(canonical_text, encoding="utf-8")

# Chuẩn bị src/ pinned tại commit efee164a857736181f9dcd0ebf023f47b11e8844
SNAPSHOT_SRC_SOURCE = "missing"
if STRUCTURED_INPUT_DIR and (STRUCTURED_INPUT_DIR / "src" / "agent" / "sandbox.py").is_file():
    dst_src = OPENCODE_RUNTIME_DIR / "src"
    remove_tree_safe(dst_src)
    shutil.copytree(STRUCTURED_INPUT_DIR / "src", dst_src)
    SNAPSHOT_SRC_SOURCE = f"structured_input:{STRUCTURED_INPUT_DIR / 'src'}"
elif REPO_DIR and (REPO_DIR / ".git").exists():
    dst_src = OPENCODE_RUNTIME_DIR / "src"
    remove_tree_safe(dst_src)
    chk = subprocess.run(
        ["git", "-C", str(REPO_DIR), "cat-file", "-t", EXPECTED_SNAPSHOT_COMMIT],
        capture_output=True, text=True, check=False,
    )
    if chk.returncode == 0 and chk.stdout.strip() == "commit":
        tar_proc = subprocess.run(
            ["git", "-C", str(REPO_DIR), "archive", "--format=tar", EXPECTED_SNAPSHOT_COMMIT, "src"],
            capture_output=True, check=False,
        )
        if tar_proc.returncode == 0:
            with tarfile.open(fileobj=io.BytesIO(tar_proc.stdout), mode="r:") as tf:
                tf.extractall(OPENCODE_RUNTIME_DIR)
            SNAPSHOT_SRC_SOURCE = f"git_archive:{EXPECTED_SNAPSHOT_COMMIT}"
elif REPO_DIR and (REPO_DIR / "src" / "agent" / "sandbox.py").is_file():
    dst_src = OPENCODE_RUNTIME_DIR / "src"
    remove_tree_safe(dst_src)
    shutil.copytree(REPO_DIR / "src", dst_src)
    SNAPSHOT_SRC_SOURCE = f"repo_fallback:{REPO_DIR / 'src'}"

STRUCTURED_SRC_ROOT = OPENCODE_RUNTIME_DIR if (OPENCODE_RUNTIME_DIR / "src" / "agent" / "sandbox.py").is_file() else REPO_DIR

# Chỉ nạp reference trong phiên score_only (tách biệt tuyệt đối khỏi phiên inference dùng run_python)
EXPOSED_REFERENCE_FILES = scan_exposed_reference_files(SEARCH_ROOTS, [SPLIT_DIR, REPO_DIR, WORK_ROOT])
REFERENCE_PATH = None
if EXECUTION_STAGE == "score_only":
    if MANUAL_PATHS["reference_file"] and Path(MANUAL_PATHS["reference_file"]).is_file():
        REFERENCE_PATH = Path(MANUAL_PATHS["reference_file"])
    elif SPLIT_DIR and (SPLIT_DIR / "answers.jsonl").is_file():
        REFERENCE_PATH = SPLIT_DIR / "answers.jsonl"
    elif REPO_DIR and (REPO_DIR / "eval_gold" / "unseen_gold_50.jsonl").is_file():
        REFERENCE_PATH = REPO_DIR / "eval_gold" / "unseen_gold_50.jsonl"
    else:
        REFERENCE_PATH = find_candidate_file(lambda p: p.name in ("answers.jsonl", "unseen_gold_50.jsonl"))
    if not REFERENCE_PATH and os.environ.get("DATAMATE_DISABLE_HF_AUTO_DOWNLOAD", "0") != "1":
        print(f"Đang tải Stage-2 reference bundle từ Hugging Face ({HF_DATASET_REPO}/{HF_REFERENCE_BUNDLE_PATH})...")
        _ref_zip = download_from_hf(HF_REFERENCE_BUNDLE_PATH)
        _extract_zip_clean(_ref_zip, HF_STAGE2_REF_DIR)
        if SPLIT == "train" and (HF_STAGE2_REF_DIR / "data" / "train" / "answers.jsonl").is_file():
            REFERENCE_PATH = HF_STAGE2_REF_DIR / "data" / "train" / "answers.jsonl"
        elif (HF_STAGE2_REF_DIR / "eval_gold" / "unseen_gold_50.jsonl").is_file():
            REFERENCE_PATH = HF_STAGE2_REF_DIR / "eval_gold" / "unseen_gold_50.jsonl"
        elif (HF_STAGE2_REF_DIR / "data" / "train" / "answers.jsonl").is_file():
            REFERENCE_PATH = HF_STAGE2_REF_DIR / "data" / "train" / "answers.jsonl"

# ------------------------------------------------------------------------------
# 4. Kiểm tra Pre-flight & Guard chống lộ đáp án
# ------------------------------------------------------------------------------
PREFLIGHT_ERRORS = []
PREFLIGHT_WARNINGS = []

if EXECUTION_STAGE not in ("inference", "score_only"):
    PREFLIGHT_ERRORS.append(f"EXECUTION_STAGE không hợp lệ: {EXECUTION_STAGE!r} (chỉ nhận 'inference' hoặc 'score_only').")

if not REPO_DIR:
    PREFLIGHT_ERRORS.append("Thiếu Repo bundle chứa src/eval/scorer.py và src/schemas.py.")
if not SPLIT_DIR and EXECUTION_STAGE == "inference":
    PREFLIGHT_ERRORS.append(f"Thiếu data split ({SPLIT}) chứa questions.jsonl, rulebook.md và thư mục data/.")
if EXECUTION_STAGE == "inference" and not MODEL_INPUT_DIR:
    PREFLIGHT_ERRORS.append(
        f"Thiếu Kaggle Input model cho {BASE_MODEL_TAG} (cần thư mục Ollama manifests/+blobs/ hoặc file .gguf). "
        f"Digest chuẩn: {EXPECTED_MODEL_DIGEST}."
    )
if not (OPENCODE_RUNTIME_DIR / "src" / "agent" / "sandbox.py").is_file():
    PREFLIGHT_ERRORS.append(
        f"Thiếu snapshot src/agent/sandbox.py (commit {EXPECTED_SNAPSHOT_COMMIT}). "
        "Hãy đóng gói tmp/opencode-structured/src/ vào Kaggle Input."
    )

# Chặn nếu Phiên inference dùng eval_agent (run_python) nhưng vẫn nhìn thấy file đáp án trên filesystem
if EXECUTION_STAGE == "inference" and TOOL_MODE == "eval_agent" and EXPOSED_REFERENCE_FILES:
    sample_refs = ", ".join(str(p) for p in EXPOSED_REFERENCE_FILES[:5])
    msg = (
        f"Phát hiện file đáp án chuẩn trên filesystem trong phiên inference có run_python: {sample_refs}. "
        "run_python chạy cùng quyền filesystem với notebook nên có thể đọc được đáp án. "
        "Hãy chạy Phiên 1 (EXECUTION_STAGE='inference') với Input đã tách riêng answers.jsonl/eval_gold, "
        "sau đó chấm PA ở Phiên 2 (EXECUTION_STAGE='score_only') — hoặc dùng TOOL_MODE='structured_only'."
    )
    if ALLOW_EXPOSED_REFERENCE_IN_RUN_PYTHON:
        PREFLIGHT_WARNINGS.append(f"[ALLOW_EXPOSED_REFERENCE_IN_RUN_PYTHON=True] {msg}")
    else:
        PREFLIGHT_ERRORS.append(msg)

if EXECUTION_STAGE == "score_only" and not REFERENCE_PATH:
    PREFLIGHT_ERRORS.append("EXECUTION_STAGE='score_only' nhưng không tìm thấy file reference (answers.jsonl hoặc unseen_gold_50.jsonl).")

try:
    test_w = RUN_DIR / ".write_probe"
    test_w.write_text("ok", encoding="utf-8")
    test_w.unlink()
except Exception as exc:
    PREFLIGHT_ERRORS.append(f"Không có quyền ghi vào {RUN_DIR}: {exc}")

if SPLIT_DIR and not os.access(SPLIT_DIR / "questions.jsonl", os.R_OK):
    PREFLIGHT_ERRORS.append(f"Không có quyền đọc {SPLIT_DIR / 'questions.jsonl'}")

if STRUCTURED_SRC_ROOT:
    sys.path.insert(0, str(STRUCTURED_SRC_ROOT))
    from src.agent.sandbox import (  # noqa: E402
        explore_data,
        load_clean_df,
        resolve_date,
        run_python_in_sandbox,
        validate_protocol_payload,
    )
if REPO_DIR and str(REPO_DIR) not in sys.path:
    sys.path.append(str(REPO_DIR))
if REPO_DIR:
    from src.eval.scorer import evaluate_predictions  # noqa: E402
    from src.schemas import AnswerType, QuestionItem, SubmissionItem, VALID_ANSWER_TYPES  # noqa: E402

MODEL_DIGEST_CHECKS = {}
if MODEL_INPUT_DIR and (MODEL_INPUT_DIR / "blobs").is_dir():
    for expected_sha in EXPECTED_BLOB_SHA256S:
        candidates = [
            MODEL_INPUT_DIR / "blobs" / f"sha256-{expected_sha}",
            MODEL_INPUT_DIR / "blobs" / f"sha256_{expected_sha}",
            MODEL_INPUT_DIR / "blobs" / expected_sha,
        ]
        matched = next((c for c in candidates if c.is_file()), None)
        if matched:
            MODEL_DIGEST_CHECKS[expected_sha] = {"present": True, "path": str(matched), "size": matched.stat().st_size}
        else:
            MODEL_DIGEST_CHECKS[expected_sha] = {"present": False, "status": "will_download_in_cell2", "path": None}
            PREFLIGHT_WARNINGS.append(
                f"Blob sha256-{expected_sha} chưa nằm sẵn trong {MODEL_INPUT_DIR / 'blobs'} — Cell 2 sẽ tự tải từ HF ({HF_OLLAMA_BLOBS_PREFIX}) hoặc ollama pull."
            )

CALENDAR_PATH = (SPLIT_DIR / "calendar.csv") if SPLIT_DIR and (SPLIT_DIR / "calendar.csv").is_file() else None
RULEBOOK_PATH = (SPLIT_DIR / "rulebook.md") if SPLIT_DIR and (SPLIT_DIR / "rulebook.md").is_file() else None
RULEBOOK_SHA256 = hashlib.sha256(RULEBOOK_PATH.read_bytes()).hexdigest() if RULEBOOK_PATH else ""

print("\n=== PREFLIGHT SUMMARY ===")
print(f"  EXECUTION_STAGE      : {EXECUTION_STAGE}")
print(f"  TOOL_MODE            : {TOOL_MODE}")
print(f"  RUN_SMOKE_TESTS      : {RUN_SMOKE_TESTS} (SINGLE_QUESTION_ONLY={SMOKE_SINGLE_QUESTION_ONLY})")
print(f"  RUN_BATCH_EVAL       : {RUN_BATCH_EVAL}")
print(f"  REPO_DIR             : {REPO_DIR}")
print(f"  SPLIT_DIR            : {SPLIT_DIR}")
print(f"  CALENDAR_PATH        : {CALENDAR_PATH}")
print(f"  RULEBOOK_PATH        : {RULEBOOK_PATH} (sha256={RULEBOOK_SHA256[:12]})")
print(f"  REFERENCE_PATH       : {REFERENCE_PATH} (chỉ nạp khi EXECUTION_STAGE=='score_only')")
print(f"  EXPOSED_REFS_FOUND   : {len(EXPOSED_REFERENCE_FILES)}")
print(f"  MODEL_INPUT_DIR      : {MODEL_INPUT_DIR}")
print(f"  OPENCODE_RUNTIME_DIR : {OPENCODE_RUNTIME_DIR}")
print(f"  SNAPSHOT_SRC_SOURCE  : {SNAPSHOT_SRC_SOURCE}")
print(f"  RUNTIME_BIN_DIR      : {RUNTIME_BIN_DIR}")
print(f"  GPUs detected        : {GPU_COUNT}")
print(f"  MANIFEST_50_IDS      : {len(MANIFEST_50_IDS)} questions")
print(f"  CSV_HOLDOUT_IDS      : {len(CSV_COUNT_DISTINCT_HOLDOUT_IDS)} questions")
if MODEL_DIGEST_CHECKS:
    print(f"  MODEL_DIGEST_CHECKS  : {json.dumps(MODEL_DIGEST_CHECKS, ensure_ascii=False)}")
for w in PREFLIGHT_WARNINGS:
    print(f"  [WARN] {w}")
if PREFLIGHT_ERRORS:
    for e in PREFLIGHT_ERRORS:
        print(f"  [ERROR] {e}")
    raise RuntimeError("Preflight thất bại — vui lòng kiểm tra các lỗi ở trên trước khi chạy.")
print("Preflight OK!")


In [ ]:
# ==============================================================================
# CELL 2: SETUP RUNTIME (OLLAMA 0.40.7 + OPENCODE 1.19.5) & TOOL SELF-TEST
# ==============================================================================
import atexit
import platform
import socket
import tarfile
import time
import urllib.request
import zipfile

SETUP_START = time.time()
OLLAMA_BIN = None
OPENCODE_BIN = None
PRE_STAGED_NODE_MODULES = None
OLLAMA_WORKERS = []
SMOKE_TAG = f"datamate-16k-t0-{RUN_TIMESTAMP}:latest"
RUNTIME_INFO = {"execution_stage": EXECUTION_STAGE}


def extract_archive(archive_path: Path, extract_dir: Path) -> None:
    """Giải nén .tar.zst, .tzst, .tgz, .tar.gz, .tar, .zip."""
    extract_dir.mkdir(parents=True, exist_ok=True)
    name_lower = archive_path.name.lower()
    if name_lower.endswith(".tar.zst") or name_lower.endswith(".tzst"):
        proc = subprocess.run(
            ["tar", "--zstd", "-xf", str(archive_path), "-C", str(extract_dir)],
            capture_output=True, text=True, check=False,
        )
        if proc.returncode == 0:
            return
        if shutil.which("zstd"):
            pipe_proc = subprocess.run(
                ["bash", "-lc", f'zstd -dc "{archive_path}" | tar -xf - -C "{extract_dir}"'],
                capture_output=True, text=True, check=False,
            )
            if pipe_proc.returncode == 0:
                return
        try:
            import zstandard as zstd  # type: ignore
            with archive_path.open("rb") as fh:
                dctx = zstd.ZstdDecompressor()
                with dctx.stream_reader(fh) as reader:
                    with tarfile.open(fileobj=reader, mode="r|") as tf:
                        tf.extractall(extract_dir)
            return
        except Exception as exc:
            raise RuntimeError(
                f"Giải nén {archive_path.name} thất bại (tar --zstd stderr={proc.stderr!r}): {exc}"
            ) from exc
    elif name_lower.endswith(".tgz") or name_lower.endswith(".tar.gz"):
        with tarfile.open(archive_path, "r:gz") as tf:
            tf.extractall(extract_dir)
    elif name_lower.endswith(".tar"):
        with tarfile.open(archive_path, "r:") as tf:
            tf.extractall(extract_dir)
    elif name_lower.endswith(".zip"):
        with zipfile.ZipFile(archive_path, "r") as zf:
            zf.extractall(extract_dir)
    else:
        raise ValueError(f"Định dạng archive chưa hỗ trợ: {archive_path}")


def _stage_extracted_libs(search_root: Path) -> None:
    lib_src = search_root / "lib"
    if not lib_src.is_dir():
        return
    lib_dst = WORK_ROOT / "lib"
    remove_tree_safe(lib_dst)
    shutil.copytree(lib_src, lib_dst)
    extra_ld = [str(lib_dst / "ollama"), str(lib_dst)]
    existing_ld = [p for p in os.environ.get("LD_LIBRARY_PATH", "").split(":") if p]
    os.environ["LD_LIBRARY_PATH"] = ":".join([*extra_ld, *existing_ld])


def check_internet(timeout=3.0) -> bool:
    try:
        socket.create_connection(("1.1.1.1", 443), timeout=timeout).close()
        return True
    except OSError:
        return False


def start_ollama_worker(worker_idx: int, port: int, cuda_visible_devices: str | None = None):
    env = os.environ.copy()
    env["OLLAMA_HOST"] = f"127.0.0.1:{port}"
    env["OLLAMA_MODELS"] = str(OLLAMA_MODELS_DIR)
    env["OLLAMA_NUM_PARALLEL"] = str(PARALLEL_SLOTS_PER_GPU)
    env["OLLAMA_MAX_LOADED_MODELS"] = "1"
    env["OLLAMA_FLASH_ATTENTION"] = "1"
    env["OLLAMA_KEEP_ALIVE"] = "15m"
    if cuda_visible_devices is not None:
        env["CUDA_VISIBLE_DEVICES"] = cuda_visible_devices
    log_path = RUN_DIR / f"ollama_worker_{worker_idx}.log"
    log_fh = open(log_path, "w", encoding="utf-8")
    proc = subprocess.Popen([OLLAMA_BIN, "serve"], env=env, stdout=log_fh, stderr=subprocess.STDOUT)

    deadline = time.time() + 60
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{port}/api/tags", timeout=2) as resp:
                if resp.status == 200:
                    return {
                        "worker_idx": worker_idx,
                        "port": port,
                        "cuda_visible_devices": cuda_visible_devices,
                        "proc": proc,
                        "log_path": str(log_path),
                    }
        except Exception:
            time.sleep(0.5)
    raise RuntimeError(f"Ollama worker {worker_idx} (port {port}) không khởi động được. Xem {log_path}")


def stop_all_ollama_workers():
    for w in OLLAMA_WORKERS:
        proc = w.get("proc")
        if proc and proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=10)
            except subprocess.TimeoutExpired:
                proc.kill()
    OLLAMA_WORKERS.clear()


atexit.register(stop_all_ollama_workers)


if EXECUTION_STAGE == "score_only":
    print("EXECUTION_STAGE='score_only' -> bỏ qua Cell 2 (setup Ollama/OpenCode).")
else:
    INTERNET_ENABLED = check_internet()
    BIN_STAGE_DIR = WORK_ROOT / "bin"
    BIN_STAGE_DIR.mkdir(parents=True, exist_ok=True)
    os.environ["PATH"] = f"{BIN_STAGE_DIR}{os.pathsep}{os.environ.get('PATH', '')}"
    os.environ["OPENCODE_DISABLE_MODELS_FETCH"] = "1"
    os.environ["OPENCODE_DISABLE_AUTOUPDATE"] = "1"

    def locate_or_install_binary(name: str, search_dir: Path | None, install_fn=None) -> str:
        if search_dir and search_dir.exists():
            for cand in [search_dir / name, search_dir / "bin" / name]:
                if cand.is_file():
                    dst = BIN_STAGE_DIR / name
                    shutil.copy2(cand, dst)
                    dst.chmod(0o755)
                    _stage_extracted_libs(search_dir)
                    return str(dst)
            for arc in sorted(search_dir.glob(f"*{name}*")):
                if arc.is_file() and any(
                    arc.name.lower().endswith(ext)
                    for ext in (".tar.zst", ".tzst", ".tgz", ".tar.gz", ".tar", ".zip")
                ):
                    extract_tmp = WORK_ROOT / f"_extract_{name}"
                    remove_tree_safe(extract_tmp)
                    extract_archive(arc, extract_tmp)
                    _stage_extracted_libs(extract_tmp)
                    found = next((p for p in extract_tmp.rglob(name) if p.is_file()), None)
                    if found:
                        dst = BIN_STAGE_DIR / name
                        shutil.copy2(found, dst)
                        dst.chmod(0o755)
                        return str(dst)
        sys_bin = shutil.which(name)
        if sys_bin:
            return sys_bin
        if install_fn is not None and INTERNET_ENABLED:
            install_fn()
            sys_bin = shutil.which(name)
            if sys_bin:
                return sys_bin
        raise RuntimeError(
            f"Không tìm thấy binary '{name}'. Hãy attach Kaggle Input chứa '{name}' "
            f"(Ollama {EXPECTED_OLLAMA_VERSION} ollama-linux-amd64.tar.zst, OpenCode {EXPECTED_OPENCODE_VERSION}) hoặc bật Internet."
        )

    def install_ollama_pinned():
        print(f"Đang cài đặt Ollama v{EXPECTED_OLLAMA_VERSION}...")
        url = f"https://github.com/ollama/ollama/releases/download/v{EXPECTED_OLLAMA_VERSION}/ollama-linux-amd64.tar.zst"
        archive_path = WORK_ROOT / "ollama-linux-amd64.tar.zst"
        urllib.request.urlretrieve(url, archive_path)
        extract_dir = WORK_ROOT / "ollama_dist"
        remove_tree_safe(extract_dir)
        extract_archive(archive_path, extract_dir)
        bin_cand = next((p for p in extract_dir.rglob("ollama") if p.is_file()), None)
        if not bin_cand:
            raise RuntimeError("Không thấy binary ollama sau khi giải nén ollama-linux-amd64.tar.zst.")
        dst = BIN_STAGE_DIR / "ollama"
        shutil.copy2(bin_cand, dst)
        dst.chmod(0o755)
        _stage_extracted_libs(extract_dir)

    def install_opencode_pinned():
        print(f"Đang cài đặt OpenCode v{EXPECTED_OPENCODE_VERSION}...")
        cmd = f"npm install -g opencode-ai@{EXPECTED_OPENCODE_VERSION} || curl -fsSL https://opencode.ai/install | bash"
        proc = subprocess.run(["bash", "-lc", cmd], capture_output=True, text=True, check=False)
        if proc.returncode != 0:
            raise RuntimeError(f"Cài OpenCode thất bại: {proc.stderr[-1000:]}")

    OLLAMA_BIN = locate_or_install_binary("ollama", RUNTIME_BIN_DIR, install_ollama_pinned)
    OPENCODE_BIN = locate_or_install_binary("opencode", RUNTIME_BIN_DIR, install_opencode_pinned)

    OLLAMA_MODELS_DIR = WORK_ROOT / "ollama_models"
    OLLAMA_MODELS_DIR.mkdir(parents=True, exist_ok=True)
    for dname in ("manifests", "manifests-v2", "metadata"):
        src_d = MODEL_INPUT_DIR / dname
        if src_d.is_dir():
            dst_d = OLLAMA_MODELS_DIR / dname
            remove_tree_safe(dst_d)
            shutil.copytree(src_d, dst_d)

    dst_blobs = OLLAMA_MODELS_DIR / "blobs"
    dst_blobs.mkdir(parents=True, exist_ok=True)
    if (MODEL_INPUT_DIR / "blobs").is_dir():
        for blob_file in (MODEL_INPUT_DIR / "blobs").iterdir():
            target = dst_blobs / blob_file.name
            if not target.exists():
                try:
                    target.symlink_to(blob_file.resolve())
                except OSError:
                    shutil.copy2(blob_file, target)

    for expected_sha in EXPECTED_BLOB_SHA256S:
        dst_blob = dst_blobs / f"sha256-{expected_sha}"
        if not dst_blob.exists() and INTERNET_ENABLED:
            print(f"Đang tải Ollama blob sha256-{expected_sha[:12]}... từ HF ({HF_DATASET_REPO})...")
            try:
                hf_blob_path = download_from_hf(f"{HF_OLLAMA_BLOBS_PREFIX}/sha256-{expected_sha}")
                try:
                    dst_blob.symlink_to(hf_blob_path.resolve())
                except OSError:
                    shutil.copy2(hf_blob_path, dst_blob)
            except Exception as exc:
                print(f"[WARN] Tải blob sha256-{expected_sha[:12]} từ HF thất bại ({exc}); sẽ thử fallback ollama pull.")

    if RUNTIME_BIN_DIR:
        for cand_nm in [RUNTIME_BIN_DIR / "node_modules", RUNTIME_BIN_DIR / "opencode_node_modules"]:
            if cand_nm.is_dir():
                PRE_STAGED_NODE_MODULES = cand_nm
                break

    def ensure_base_model_on_worker(port: int):
        with urllib.request.urlopen(f"http://127.0.0.1:{port}/api/tags", timeout=10) as resp:
            tags = [m["name"] for m in json.load(resp).get("models", [])]
        if BASE_MODEL_TAG in tags:
            return
        gguf_files = sorted(MODEL_INPUT_DIR.glob("*.gguf"))
        if not gguf_files:
            if INTERNET_ENABLED:
                print(f"Chưa thấy {BASE_MODEL_TAG} trong /api/tags -> fallback `ollama pull ornith-1.5:9b` rồi khôi phục manifest overlay...")
                pull_env = os.environ.copy()
                pull_env["OLLAMA_HOST"] = f"127.0.0.1:{port}"
                pull_env["OLLAMA_MODELS"] = str(OLLAMA_MODELS_DIR)
                subprocess.run([OLLAMA_BIN, "pull", "ornith-1.5:9b"], env=pull_env, check=True)
                for dname in ("manifests", "manifests-v2", "metadata"):
                    src_d = MODEL_INPUT_DIR / dname
                    if src_d.is_dir():
                        dst_d = OLLAMA_MODELS_DIR / dname
                        remove_tree_safe(dst_d)
                        shutil.copytree(src_d, dst_d)
                if (MODEL_INPUT_DIR / "blobs").is_dir():
                    for blob_file in (MODEL_INPUT_DIR / "blobs").iterdir():
                        target = dst_blobs / blob_file.name
                        if not target.exists():
                            shutil.copy2(blob_file, target)
                with urllib.request.urlopen(f"http://127.0.0.1:{port}/api/tags", timeout=10) as resp2:
                    tags2 = [m["name"] for m in json.load(resp2).get("models", [])]
                if BASE_MODEL_TAG in tags2:
                    return
            raise RuntimeError(f"Không thấy {BASE_MODEL_TAG} trong /api/tags và không có file .gguf để import.")
        modelfile_src = next((MODEL_INPUT_DIR / n for n in ["Modelfile", "modelfile"] if (MODEL_INPUT_DIR / n).is_file()), None)
        modelfile_tmp = WORK_ROOT / "Modelfile.generated"
        if modelfile_src:
            mf_text = modelfile_src.read_text(encoding="utf-8")
            lines = [f"FROM {gguf_files[0]}" if l.strip().upper().startswith("FROM ") else l for l in mf_text.splitlines()]
            modelfile_tmp.write_text("\n".join(lines) + "\n", encoding="utf-8")
        else:
            modelfile_tmp.write_text(
                f"FROM {gguf_files[0]}\n"
                f"PARAMETER num_ctx 8192\n"
                f"PARAMETER seed {SEED}\n"
                f"PARAMETER num_batch {NUM_BATCH}\n"
                f"PARAMETER presence_penalty {PRESENCE_PENALTY}\n",
                encoding="utf-8",
            )
        env = os.environ.copy()
        env["OLLAMA_HOST"] = f"127.0.0.1:{port}"
        env["OLLAMA_MODELS"] = str(OLLAMA_MODELS_DIR)
        subprocess.run([OLLAMA_BIN, "create", BASE_MODEL_TAG, "-f", str(modelfile_tmp)], env=env, check=True)

    def create_16k_tag(port: int, tag_name: str, temperature: float):
        payload = {
            "model": tag_name,
            "from": BASE_MODEL_TAG,
            "parameters": {
                "num_ctx": NUM_CTX,
                "temperature": temperature,
                "seed": SEED,
                "num_batch": NUM_BATCH,
                "presence_penalty": PRESENCE_PENALTY,
            },
            "stream": False,
        }
        req = urllib.request.Request(
            f"http://127.0.0.1:{port}/api/create",
            data=json.dumps(payload).encode("utf-8"),
            headers={"Content-Type": "application/json"},
        )
        with urllib.request.urlopen(req, timeout=60) as resp:
            res = json.load(resp)
        if res.get("status") != "success":
            raise RuntimeError(f"Không tạo được tag 16K {tag_name}: {res}")

    def inspect_ollama_show(port: int, model_name: str) -> dict:
        req = urllib.request.Request(
            f"http://127.0.0.1:{port}/api/show",
            data=json.dumps({"model": model_name}).encode("utf-8"),
            headers={"Content-Type": "application/json"},
        )
        with urllib.request.urlopen(req, timeout=30) as resp:
            info = json.load(resp)
        tpl = info.get("template", "")
        tpl_sha = hashlib.sha256(tpl.encode("utf-8")).hexdigest() if tpl else ""
        return {
            "template_sha256": tpl_sha,
            "matches_expected_template": tpl_sha == EXPECTED_TEMPLATE_SHA256,
            "parameters": info.get("parameters", ""),
            "details": info.get("details", {}),
        }

    stop_all_ollama_workers()
    w0_cuda = "0" if GPU_COUNT >= 1 else None
    OLLAMA_WORKERS.append(start_ollama_worker(0, 11434, cuda_visible_devices=w0_cuda))
    ensure_base_model_on_worker(11434)
    SHOW_INFO = inspect_ollama_show(11434, BASE_MODEL_TAG)
    SETUP_TIME_SEC = round(time.time() - SETUP_START, 2)

    MODEL_LOAD_START = time.time()
    create_16k_tag(11434, SMOKE_TAG, temperature=0.0)
    warmup_req = urllib.request.Request(
        "http://127.0.0.1:11434/api/generate",
        data=json.dumps({"model": SMOKE_TAG, "prompt": "OK", "stream": False, "options": {"num_predict": 1}}).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(warmup_req, timeout=180) as resp:
        _ = json.load(resp)
    MODEL_LOAD_TIME_SEC = round(time.time() - MODEL_LOAD_START, 2)

    import numpy as np
    import pandas as pd

    opencode_ver = subprocess.run([OPENCODE_BIN, "--version"], capture_output=True, text=True, check=False).stdout.strip()
    ollama_ver = subprocess.run([OLLAMA_BIN, "--version"], capture_output=True, text=True, check=False).stdout.strip()

    RUNTIME_INFO = {
        "execution_stage": EXECUTION_STAGE,
        "internet_enabled": INTERNET_ENABLED,
        "opencode_bin": OPENCODE_BIN,
        "opencode_version": opencode_ver,
        "ollama_bin": OLLAMA_BIN,
        "ollama_version": ollama_ver,
        "python_version": platform.python_version(),
        "pandas_version": pd.__version__,
        "numpy_version": np.__version__,
        "setup_time_sec": SETUP_TIME_SEC,
        "model_load_time_sec": MODEL_LOAD_TIME_SEC,
        "num_ctx_per_slot": NUM_CTX,
        "parallel_slots_per_gpu": PARALLEL_SLOTS_PER_GPU,
        "ollama_kv_ctx": NUM_CTX,
        "base_model_show": SHOW_INFO,
        "gpu_lines": GPU_LINES,
    }
    print("\n=== RUNTIME INFO ===")
    print(json.dumps(RUNTIME_INFO, indent=2, ensure_ascii=False))

    # Self-test 4 tool trên bản copy read-only của biz_125 (khớp đúng cấu trúc shop root thật)
    SELF_TEST_ROOT = WORK_ROOT / "_self_test_biz_125" / "data"
    remove_tree_safe(SELF_TEST_ROOT.parent)
    SELF_TEST_ROOT.mkdir(parents=True, exist_ok=True)
    biz_125_src = SPLIT_DIR / "data" / "biz_125"
    for item in biz_125_src.iterdir():
        if item.is_file() and (item.suffix.lower() in {".csv", ".xlsx", ".xls", ".sqlite", ".db", ".sqlite3"} or item.name.lower() == "readme.md"):
            dst = SELF_TEST_ROOT / item.name
            shutil.copy2(item, dst)
            os.chmod(dst, 0o444)

    for _tok_key in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HF_HUB_TOKEN"):
        os.environ.pop(_tok_key, None)
    HF_TOKEN_RUNTIME = ""

    bridge_env = {
        **os.environ,
        "DATAMATE_DATA_ROOT": str(SELF_TEST_ROOT),
        "DATAMATE_SRC_ROOT": str(STRUCTURED_SRC_ROOT),
        "DATAMATE_WORK_DIR": str(WORK_ROOT / "_self_test_work"),
        "DATAMATE_AS_OF_DATE": "2025-03-28",
        "DATAMATE_RULEBOOK_PATH": str(RULEBOOK_PATH),
        "DATAMATE_CALENDAR_PATH": str(CALENDAR_PATH) if CALENDAR_PATH else "",
    }
    for _tok_key in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HF_HUB_TOKEN"):
        bridge_env.pop(_tok_key, None)

    def run_bridge_self_test(action: str, payload: dict) -> dict:
        proc = subprocess.run(
            [sys.executable, str(OPENCODE_RUNTIME_DIR / "bridge.py"), action, json.dumps(payload, ensure_ascii=False)],
            env=bridge_env, capture_output=True, text=True, check=True, timeout=30,
        )
        return json.loads(proc.stdout.strip())

    assert (SELF_TEST_ROOT / "README.md").read_text(encoding="utf-8"), "read README.md failed"
    st_schema = run_bridge_self_test("inspect_table", {"file": None, "column": None})
    assert st_schema.get("file") == "ban_hang.csv", f"inspect_table schema failed: {st_schema}"
    st_labels = run_bridge_self_test("inspect_table", {"file": "ban_hang.csv", "column": "Trạng thái xử lý"})
    assert "Huỷ đơn" in st_labels.get("unique_values", []), f"inspect_table labels failed: {st_labels}"
    st_null_col = run_bridge_self_test("inspect_table", {"file": "ban_hang.csv", "column": "None"})
    assert "columns" in st_null_col, f"inspect_table column='None' normalization failed: {st_null_col}"
    st_agg = run_bridge_self_test("aggregate", {
        "file": "ban_hang.csv",
        "operation": "count_distinct",
        "date_column": "Ngày bán",
        "date_expression": "ngày hôm nay",
        "as_of_date": "2025-03-28",
        "column": "Số phiếu",
        "status_column": "Trạng thái xử lý",
        "exclude_values": ["Huỷ đơn", "Đã trả hàng"],
    })
    assert st_agg.get("answer") == 41, f"aggregate failed: {st_agg}"
    st_py = run_bridge_self_test("run_python", {"code": "emit_final(41, type='number', evidence='self_test')"})
    assert st_py.get("success") is True and st_py.get("parsed_output", {}).get("answer") == 41, f"run_python failed: {st_py}"
    print("Self-test 4 tools (read / inspect_table / aggregate / run_python): PASS!")

    DISCOVERY_START = time.time()
    _disc_dir = WORK_ROOT / "_opencode_discovery_probe"
    remove_tree_safe(_disc_dir)
    (_disc_dir / "tools").mkdir(parents=True, exist_ok=True)
    shutil.copy2(OPENCODE_RUNTIME_DIR / "run.js", _disc_dir / "run.js")
    for _tname in ("inspect_table.js", "aggregate.js", "run_python.js", "explore_data.js"):
        shutil.copy2(OPENCODE_RUNTIME_DIR / "tools" / _tname, _disc_dir / "tools" / _tname)
    if PRE_STAGED_NODE_MODULES and PRE_STAGED_NODE_MODULES.is_dir():
        try:
            (_disc_dir / "node_modules").symlink_to(PRE_STAGED_NODE_MODULES.resolve())
        except OSError:
            shutil.copytree(PRE_STAGED_NODE_MODULES, _disc_dir / "node_modules")
    _disc_env = {
        **os.environ,
        "HOME": str(_disc_dir),
        "OPENCODE_CONFIG_DIR": str(_disc_dir),
        "OPENCODE_DISABLE_MODELS_FETCH": "1",
        "OPENCODE_DISABLE_AUTOUPDATE": "1",
    }
    _disc_proc = subprocess.run([OPENCODE_BIN, "--pure", "debug", "config"], env=_disc_env, capture_output=True, text=True, check=False, timeout=120)
    RUNTIME_INFO["tool_discovery_check"] = {
        "returncode": _disc_proc.returncode,
        "elapsed_sec": round(time.time() - DISCOVERY_START, 2),
        "pre_staged_node_modules": str(PRE_STAGED_NODE_MODULES) if PRE_STAGED_NODE_MODULES else None,
    }
    print("OpenCode tool discovery check:", RUNTIME_INFO["tool_discovery_check"])


In [ ]:
# ==============================================================================
# CELL 3: AGENT RUNNER, SHOP ROOT CÔ LẬP & CHUẨN HÓA FINAL THEO MỤC 4
# ==============================================================================
import math
import re
import threading
import time
import unicodedata
from concurrent.futures import ThreadPoolExecutor

ALLOWED_SHOP_EXTENSIONS = {".csv", ".xlsx", ".xls", ".sqlite", ".db", ".sqlite3"}


def _remove_readonly_tree(target: Path) -> None:
    if not target.exists():
        return
    for root, dirs, files in os.walk(target):
        for d in dirs:
            try:
                os.chmod(Path(root) / d, 0o755)
            except OSError:
                pass
        for f in files:
            try:
                os.chmod(Path(root) / f, 0o644)
            except OSError:
                pass
    shutil.rmtree(target)


def prepare_isolated_shop_root(question_item: QuestionItem, split_dir: Path, root_base: Path) -> Path:
    """Copy dữ liệu shop vào shop root riêng (không dùng symlink để không trượt guard resolve() của bridge.py)."""
    src_shop = split_dir / "data" / question_item.business_id
    if not src_shop.is_dir():
        raise FileNotFoundError(f"Không thấy thư mục shop {src_shop}")

    shop_root = root_base / question_item.id / "data"
    _remove_readonly_tree(shop_root.parent)
    shop_root.mkdir(parents=True, exist_ok=True)

    for entry in sorted(src_shop.iterdir()):
        if entry.is_file() and (entry.suffix.lower() in ALLOWED_SHOP_EXTENSIONS or entry.name.lower() == "readme.md"):
            try:
                repaired = entry.name.encode("cp437").decode("utf-8")
            except (UnicodeEncodeError, UnicodeDecodeError):
                repaired = ""
            target_name = (
                repaired
                if repaired and repaired != entry.name and Path(repaired).name == repaired and not (src_shop / repaired).exists()
                else entry.name
            )
            dst = shop_root / target_name
            if not dst.exists():
                shutil.copy2(entry, dst)
                os.chmod(dst, 0o444)
    return shop_root


def build_system_instructions(shop_root: Path, tool_mode: str) -> str:
    """Tạo system instructions khớp mục 2 của handoff, thay /data bằng shop_root thực tế."""
    if tool_mode == "eval_agent":
        available_files = (
            sorted(p.name for p in shop_root.iterdir() if p.is_file() and p.name.lower() != "readme.md")
            if shop_root.exists()
            else []
        )
        readme_path = shop_root / "README.md"
        files_hint = f" Danh sách file dữ liệu trong shop: {available_files}." if available_files else ""
        return (
            f"Dữ liệu shop nằm trong `{shop_root}`.{files_hint}\n"
            "QUY TRÌNH BẮT BUỘC (thực hiện ĐÚNG THỨ TỰ từng bước, KHÔNG nhảy cóc vào `run_python` khi chưa xong Bước 1 và Bước 2):\n"
            f"- Bước 1 — ĐỌC THÔNG TIN SHOP & RULEBOOK TRƯỚC: Trước khi viết code hay tính toán, đọc kỹ rulebook ở trên trong system prompt để xác định định nghĩa chỉ tiêu, quy ước làm tròn, loại trừ đơn hủy/trả và kiểu `type` cần trả; đồng thời gọi tool `read` với `filePath=\"{readme_path}\"` để nắm quy ước riêng của shop, ý nghĩa từng file và từng cột.\n"
            "- Bước 2 — XEM BẢNG & SHEET BẰNG `inspect_table`: Gọi `inspect_table(file=None, column=None)` và `inspect_table(file='<ten_file>', column=None)` cho MỌI file trong danh sách (không bỏ file nào, kể cả file tên khó hiểu như sổ quỹ, phiếu thu chi: công nợ, thanh toán, giá, tồn thường nằm ở file riêng; chưa mở hết thì không được kết luận là thiếu dữ liệu) để biết chính xác danh sách sheet (chọn ĐÚNG sheet khớp tháng/năm của câu hỏi, không đọc nhầm sheet khác), tên cột, và nhãn `Trạng thái` thực tế.\n"
            "- Bước 3 — XEM DỮ LIỆU Ở ĐÚNG MỐC THỜI GIAN CỦA CÂU HỎI BẰNG TOOL `explore_data` (bắt buộc với câu có ngày/tuần/tháng/quý hoặc cụm như \'hôm qua\', \'tháng này\'): gọi tool `explore_data(file=\'<ten_file>\', date=\'<mốc>\')` (không cần viết code; thêm `sheet=\'<ten_sheet>\'` nếu file nhiều sheet). `date` nhận \'YYYY-MM-DD\', \'YYYY-MM\', \'dd/mm/yyyy\' hoặc nguyên cụm như \'hôm qua\'. Đọc `date_column`, `rows_in_window`, `status_counts_in_window`, `sample_rows`, đặc biệt `summary_rows_near_window` và `notes`: đó là dòng tổng/cộng do chủ shop ghi sẵn (ngày trống, nội dung \'Tổng ...\'). TUYỆT ĐỐI không cộng chúng chung với dòng hàng, nếu không kết quả bị gấp đôi; chỉ tính trên dòng hàng thật.\n"
            "- Bước 4 — TÍNH TOÁN & CHỐT ĐÁP ÁN:\n"
            "  + Nếu chỉ cần đếm `count_distinct` trên một file CSV đơn giản: dùng `aggregate`.\n"
            "  + Với mọi câu hỏi còn lại (tra giá, lọc nhiều điều kiện, tổng/trung bình/tỷ lệ, lập bảng/danh sách, nối nhiều bảng, đọc `.xlsx`/`.sqlite`): dùng `run_python`.\n"
            "Quy tắc BẮT BUỘC khi gọi `run_python`:\n"
            "  (1) KHÔNG hardcode `/data/...`; truyền tên file tương đối cho `pd.read_excel`, `pd.read_csv`, `load_clean_df('ten_file', dedup_order=False)` (không truyền `parse_dates`), hoặc `explore_data('ten_file')`.\n"
            "  (2) Lọc ngày theo `resolve_date(expr, as_of_date=AS_OF_DATE)` + `parse_vn_date(series)` và chuẩn hóa mọi giá/tiền bằng `parse_vn_number(val)` (KHÔNG dùng `int()`/`float()` trực tiếp lên chuỗi `'37,000.0000'`); nếu rulebook loại đơn hủy/trả thì loại toàn bộ các dòng thuộc các mã đơn có trạng thái đó.\n"
            "  (3) Các helper `explore_data`, `load_clean_df`, `resolve_date`, `parse_vn_date`, `parse_vn_number`, `vn_num`, `emit_obs`, `emit_final` ĐÃ CÓ SẴN trong globals (KHÔNG import). Mọi lượt `run_python` PHẢI kết thúc bằng `emit_obs(data, msg)` khi thăm dò hoặc `emit_final(answer, type=\"number\", evidence=\"...\")` khi chốt (`type` đặt trong ngoặc kép hoặc dùng identifier `number`/`string`/`boolean`/`table`/`list`, KHÔNG gọi `emit_final()` rỗng).\n"
            "  (4) Với file XLSX nhiều sheet, mở ĐÚNG sheet khớp mốc thời gian trong câu hỏi (ví dụ ngày 01/12/2025 -> sheet `T12_2025`) và dùng `usecols` nếu file lớn.\n"
            "  (5) Chỉ chốt đáp án cuối sau khi `run_python` trả `kind=final` (hoặc `aggregate` hợp lệ). Sau khi gọi `emit_final` thành công, BẮT BUỘC phải xuất đáp án cuối cùng trong câu trả lời (trả đúng giá trị JSON của `answer` ví dụ `42`, bỏ giải thích). Nếu chưa xác minh được thì trả {\"type\":\"abstain\",\"answer\":null}.\n"
            "  (6) TRƯỚC KHI CHỐT: trong cùng lượt code, kiểm tra lại điều kiện đã áp dụng (kỳ thời gian, trạng thái, dòng tổng, số dòng trước/sau mỗi bộ lọc), đối chiếu từng mục liên quan của rulebook (kể cả mục cách ghi dữ liệu trong file) và so bậc độ lớn với một kỳ liền kề cùng loại; nếu đáp án lệch bất thường (ví dụ gấp nhiều lần kỳ liền kề) thì `emit_obs` để điều tra thay vì `emit_final`. Ghi các điều kiện đã áp dụng vào `evidence`.\n"
            "  (7) KHÔNG sa vào suy nghĩ dài: khi phân vân định nghĩa, chọn định nghĩa theo rulebook rồi chạy code kiểm tra bằng dữ liệu thật; mỗi lượt suy luận phải kết thúc bằng một lệnh gọi tool hoặc đáp án.\n"
        )

    readme_hint = f"Đọc {shop_root / 'README.md'} để hiểu quy ước của shop. "
    answer_hint = "Trả đúng một JSON có hai trường type và answer từ kết quả tool. "
    return (
        f"Tuân thủ rulebook của split và dùng as_of_date từ đầu vào. "
        f"Dùng inspect_table để xác nhận cột và nhãn thực tế dùng khi lọc. "
        f"Dùng aggregate để tính dựa trên các cột, ngày và quy tắc vừa xác định. "
        f"{answer_hint}"
        f'Nếu chưa xác minh được, trả {{"type":"abstain","answer":null}}. '
        f"Dữ liệu nằm trong {shop_root}. "
        f"{readme_hint}"
        f"Gọi inspect_table với file và column đều là null để tìm bảng, rồi đọc nhãn từ cột trạng thái."
    )


def build_user_prompt(question_item: QuestionItem, shop_root: Path, tool_mode: str) -> str:
    prompt = f"{question_item.question}\n\nas_of_date: {question_item.as_of_date}"
    if tool_mode == "auto_readme_schema":
        readme = (shop_root / "README.md").read_text(encoding="utf-8") if (shop_root / "README.md").is_file() else ""
        prompt += f"\n\nREADME của shop:\n{readme}"
    return prompt


def setup_opencode_session_dir(session_dir: Path, tool_mode: str) -> Path:
    """Chuẩn bị OPENCODE_CONFIG_DIR chứa tools/, run.js và node_modules pre-staged (nếu có)."""
    if session_dir.exists():
        shutil.rmtree(session_dir)
    session_dir.mkdir(parents=True, exist_ok=True)
    tools_dst = session_dir / "tools"
    tools_dst.mkdir(parents=True, exist_ok=True)

    run_js_dst = session_dir / "run.js"
    shutil.copy2(OPENCODE_RUNTIME_DIR / "run.js", run_js_dst)

    allowed_tools = ["inspect_table.js", "aggregate.js"]
    if tool_mode == "eval_agent":
        allowed_tools += ["run_python.js", "explore_data.js"]
    for fname in allowed_tools:
        shutil.copy2(OPENCODE_RUNTIME_DIR / "tools" / fname, tools_dst / fname)

    if PRE_STAGED_NODE_MODULES and PRE_STAGED_NODE_MODULES.is_dir():
        nm_dst = session_dir / "node_modules"
        try:
            nm_dst.symlink_to(PRE_STAGED_NODE_MODULES.resolve())
        except OSError:
            shutil.copytree(PRE_STAGED_NODE_MODULES, nm_dst)
    return session_dir


def parse_opencode_jsonl_events(stdout_text: str, sidecar_traces: list | None = None) -> dict:
    """Tách raw_final_text theo messageID phản hồi cuối, tool_calls, tokens, finish_reason và errors."""
    events = []
    parse_errors = []
    for line in stdout_text.splitlines():
        s = line.strip()
        if not s:
            continue
        try:
            events.append(json.loads(s))
        except json.JSONDecodeError as exc:
            parse_errors.append({"line": s[:300], "error": str(exc)})

    step_starts = []
    step_finishes = []
    text_parts = []
    tool_calls = []
    api_errors = []

    for ev in events:
        etype = ev.get("type")
        part = ev.get("part") if isinstance(ev.get("part"), dict) else {}
        if etype == "step_start":
            step_starts.append(part)
        elif etype == "step_finish":
            step_finishes.append(part)
        elif etype == "text":
            text_parts.append(part)
        elif etype == "tool_use":
            state = part.get("state") if isinstance(part.get("state"), dict) else {}
            tool_calls.append({
                "tool": part.get("tool"),
                "callID": part.get("callID"),
                "messageID": part.get("messageID"),
                "status": state.get("status"),
                "input": state.get("input"),
                "output": state.get("output"),
                "error": state.get("error"),
                "metadata": state.get("metadata"),
                "time": state.get("time"),
            })
        elif etype == "error":
            api_errors.append(ev)

    if sidecar_traces:
        sc_idx = 0
        for tc in tool_calls:
            tname = tc.get("tool")
            if tname not in ("inspect_table", "aggregate", "run_python", "explore_data"):
                continue
            for j in range(sc_idx, len(sidecar_traces)):
                sc = sidecar_traces[j]
                if sc.get("action") == tname and (sc.get("args") == tc.get("input") or tc.get("input") is None):
                    tc["full_result"] = sc.get("result")
                    sc_idx = j + 1
                    break
            else:
                for j in range(sc_idx, len(sidecar_traces)):
                    sc = sidecar_traces[j]
                    if sc.get("action") == tname:
                        tc["full_result"] = sc.get("result")
                        sc_idx = j + 1
                        break

    final_message_id = None
    finish_reason = None
    if step_finishes:
        last_fin = step_finishes[-1]
        final_message_id = last_fin.get("messageID")
        finish_reason = last_fin.get("reason")
    elif step_starts:
        final_message_id = step_starts[-1].get("messageID")
    elif text_parts:
        final_message_id = text_parts[-1].get("messageID")

    if final_message_id is not None:
        raw_final_text = "".join(
            str(p.get("text", ""))
            for p in text_parts
            if p.get("messageID") == final_message_id
        ).strip()
    else:
        raw_final_text = ""

    token_summary = {"input": 0, "output": 0, "reasoning": 0, "total": 0}
    for sf in step_finishes:
        tok = sf.get("tokens") or {}
        for k in ("input", "output", "reasoning", "total"):
            if isinstance(tok.get(k), (int, float)):
                token_summary[k] += int(tok[k])

    return {
        "session_id": next((ev["sessionID"] for ev in events if ev.get("sessionID")), None),
        "raw_final_text": raw_final_text,
        "final_message_id": final_message_id,
        "finish_reason": finish_reason,
        "tool_calls": tool_calls,
        "token_summary": token_summary,
        "api_errors": api_errors,
        "jsonl_parse_errors": parse_errors,
        "event_count": len(events),
    }


def _strip_thinking_blocks(text: str) -> str:
    cleaned = re.sub(r"<think>.*?</think>", "", text or "", flags=re.DOTALL | re.IGNORECASE)
    if "</think>" in cleaned.lower():
        parts = re.split(r"</think>", cleaned, flags=re.IGNORECASE)
        cleaned = parts[-1]
    return cleaned.strip()


def _strip_markdown_scalar_wrappers(text: str) -> str:
    s = (text or "").strip()
    if len(s) >= 2 and s[0] == "`" and s[-1] == "`":
        s = s[1:-1].strip()
    if len(s) >= 4 and s.startswith("**") and s.endswith("**"):
        s = s[2:-2].strip()
    elif len(s) >= 2 and s.startswith("*") and s.endswith("*"):
        s = s[1:-1].strip()
    s = re.sub(r"\s*\.$", "", s).strip()
    if len(s) >= 2 and s[0] == "`" and s[-1] == "`":
        s = s[1:-1].strip()
    return s


def _try_parse_json_candidates(text: str):
    cleaned = _strip_thinking_blocks(text)
    if not cleaned:
        return []
    candidates = []
    try:
        candidates.append(json.loads(cleaned))
    except json.JSONDecodeError:
        pass
    for m in re.finditer(r"```(?:json)?\s*(.*?)\s*```", cleaned, flags=re.DOTALL | re.IGNORECASE):
        try:
            candidates.append(json.loads(m.group(1).strip()))
        except json.JSONDecodeError:
            pass
    # Scan top-level JSON objects/arrays in cleaned using JSONDecoder (handles nested JSON safely)
    decoder = json.JSONDecoder()
    decoded_blocks = []
    idx = 0
    n = len(cleaned)
    while idx < n:
        ch = cleaned[idx]
        if ch in "{[":
            try:
                obj, end_idx = decoder.raw_decode(cleaned, idx)
                decoded_blocks.append(obj)
                idx = end_idx
                continue
            except json.JSONDecodeError:
                pass
        idx += 1
    for obj in reversed(decoded_blocks):
        candidates.append(obj)
    if not candidates:
        unwrapped = _strip_markdown_scalar_wrappers(cleaned)
        if unwrapped and unwrapped != cleaned:
            try:
                candidates.append(json.loads(unwrapped))
            except json.JSONDecodeError:
                pass
    return candidates


_ABSTAIN_CUE_RE = re.compile(
    r"(?:\babstain\b|không\s*thể|không\s*đủ|thiếu\s*dữ\s*liệu|không\s*xác\s*định|chưa\s*xác\s*minh|\bcannot\b|\bunable\s+to\b|\binsufficient\b)",
    flags=re.IGNORECASE,
)


_STEP_LIMIT_PREFIX_RE = re.compile(
    r"^\s*(?:Do not invoke any tools\.\s*)?(?:The maximum number of steps[^.\n]*\.\s*(?:No further tool calls[^.\n]*\.\s*)?(?:This constraint takes priority[^.\n]*\.\s*)?|I can no longer use tools[^.\n]*\.\s*|---\s*)*",
    flags=re.IGNORECASE,
)


def _strip_step_limit_header(text: str) -> str:
    cleaned = _strip_thinking_blocks(text)
    cleaned = re.sub(
        r"(?im)^\s*(?:Do not invoke any tools\.|The maximum number of steps[^\n]*|I can no longer use tools[^\n]*|No further tool calls will be made\.[^\n]*|\*\*Báo cáo tiến độ\s*-\s*Đã đạt số bước cực đại\*\*)\s*$",
        "",
        cleaned,
    )
    cleaned = _STEP_LIMIT_PREFIX_RE.sub("", cleaned).strip()
    return cleaned


def _extract_verified_scalar_from_prose(text: str, verified_results: list[dict], max_len: int = 400):
    """Chỉ tách scalar khi text gọn, không có dấu hiệu abstain, không mơ hồ và khớp kết quả tool đã xác minh."""
    if not verified_results:
        return False, None
    cleaned = _strip_step_limit_header(text)
    if not cleaned or len(cleaned) > max_len or _ABSTAIN_CUE_RE.search(cleaned):
        return False, None

    answer_lines = re.findall(
        r"^[ \t]*(?:\*\*)?(?:trả[ \t]+lời|đáp[ \t]*án|kết[ \t]*quả|answer|result)(?:\*\*)?[ \t]*[:\-](?:\*\*)?[ \t]*([^\r\n]*)",
        cleaned,
        flags=re.IGNORECASE | re.MULTILINE,
    )
    if answer_lines:
        if len(answer_lines) != 1:
            return False, None
        cleaned = _strip_markdown_scalar_wrappers(answer_lines[0])

    # Loại chuỗi ngày ISO YYYY-MM-DD trước khi đếm số để tránh coi ngày là số xung đột
    no_dates = re.sub(r"\b\d{4}-\d{2}-\d{2}\b", " ", cleaned)
    raw_nums = re.findall(r"(?<![A-Za-z0-9_])[-+]?\d+(?:\.\d+)?(?![A-Za-z0-9_])", no_dates)
    num_vals = []
    for tok in raw_nums:
        try:
            val = float(tok) if "." in tok else int(tok)
            if val not in num_vals:
                num_vals.append(val)
        except ValueError:
            pass
    if len(num_vals) == 1:
        v = num_vals[0]
        if any(_values_match(v, r["answer"]) for r in verified_results):
            return True, v

    cand_str = _strip_markdown_scalar_wrappers(cleaned)
    if cand_str and any(isinstance(r["answer"], str) and _values_match(cand_str, r["answer"]) for r in verified_results):
        return True, cand_str
    return False, None


def _values_match(a, b) -> bool:
    if type(a) is bool or type(b) is bool:
        return type(a) is bool and type(b) is bool and a == b
    if isinstance(a, (int, float)) and isinstance(b, (int, float)):
        if math.isnan(float(a)) or math.isnan(float(b)):
            return False
        return round(float(a), 2) == round(float(b), 2)
    if isinstance(a, str) and isinstance(b, str):
        return unicodedata.normalize("NFC", a.strip()) == unicodedata.normalize("NFC", b.strip())
    return a == b


def _validate_prediction_schema(ans_type: str, ans_val) -> tuple[bool, object, str]:
    clean_type = str(ans_type).strip().lower()
    if clean_type not in VALID_ANSWER_TYPES:
        return False, None, f"invalid_type:{clean_type}"
    if clean_type in (AnswerType.ABSTAIN.value, AnswerType.UNANSWERABLE.value):
        return True, None, ""
    probe = {
        "kind": "final",
        "type": clean_type,
        "answer": ans_val,
        "evidence": "normalized_final",
    }
    try:
        _, validated = validate_protocol_payload(probe)
        return True, validated.get("answer"), ""
    except Exception as exc:
        return False, None, f"schema_error:{exc}"


def _looks_like_wrapped_payload(ans_type: str, ans_val) -> bool:
    """Model đôi khi nhét dict/JSON vào answer kiểu string; đó là đáp án sai format, không dùng làm kết quả đã xác minh."""
    if str(ans_type).strip().lower() != "string" or not isinstance(ans_val, str):
        return False
    t = ans_val.strip()
    return t[:1] in ("{", "[") and t[-1:] in ("}", "]")


def _digits_only(text: str) -> str:
    return re.sub(r"\D", "", text or "")


def _rescue_consistent_with_prose(raw_text: str, rescue_pred: dict) -> bool:
    """Final text không parse được: chỉ cứu kết quả run_python(final) cho unanswerable, date, string khi văn bản không mâu thuẫn.
    number/percent/list/table/boolean không cứu ở đây (đã có _extract_verified_scalar_from_prose, cố ý chặt)."""
    if not rescue_pred:
        return False
    r_type = str(rescue_pred.get("type", "")).strip().lower()
    r_ans = rescue_pred.get("answer")
    if r_type == AnswerType.UNANSWERABLE.value:
        return True
    if r_ans is None:
        return False
    cleaned = _strip_step_limit_header(raw_text)
    if not cleaned or len(cleaned) > 800 or _ABSTAIN_CUE_RE.search(cleaned):
        return False
    if r_type == "date" and isinstance(r_ans, str):
        found = []
        for m in re.finditer(r"\b(\d{4})-(\d{2})-(\d{2})\b|\b(\d{1,2})/(\d{1,2})/(\d{4})\b", cleaned):
            if m.group(1):
                found.append(f"{m.group(1)}-{m.group(2)}-{m.group(3)}")
            else:
                found.append(f"{int(m.group(6)):04d}-{int(m.group(5)):02d}-{int(m.group(4)):02d}")
        return bool(found) and all(x == r_ans for x in found)
    if r_type == "string" and isinstance(r_ans, str):
        norm = lambda x: unicodedata.normalize("NFC", x).strip().casefold()
        if norm(r_ans) not in norm(cleaned):
            return False
        return all(norm(b) == norm(r_ans) for b in re.findall(r"\*\*(.+?)\*\*", cleaned))
    return False


def _coerce_month_string(typ: str, ans):
    """Model hay emit tháng dạng string "8/2025"; luật Mục 4: câu hỏi theo tháng là date "YYYY-MM"."""
    if typ == "string" and isinstance(ans, str):
        m = re.fullmatch(r"\s*(?:th[áa]ng\s*)?(\d{1,2})\s*[/-]\s*(\d{4})\s*", ans, flags=re.I)
        if m and 1 <= int(m[1]) <= 12:
            return "date", f"{m[2]}-{int(m[1]):02d}"
        m = re.fullmatch(r"\s*(\d{4})\s*[/-]\s*(\d{1,2})\s*", ans)   # đã đúng dạng YYYY-MM nhưng gắn type string
        if m and 1 <= int(m[2]) <= 12:
            return "date", f"{m[1]}-{int(m[2]):02d}"
    return typ, ans


def _extract_verified_tool_results_chronological(tool_calls: list) -> list[dict]:
    """Trả danh sách kết quả tính hợp lệ từ aggregate và run_python(final) đúng theo thứ tự thực thi."""
    verified = []
    for idx, tc in enumerate(tool_calls):
        if tc.get("status") != "completed":
            continue
        payload = tc.get("full_result")
        if payload is None:
            raw_out = tc.get("output")
            if isinstance(raw_out, str) and raw_out.strip():
                try:
                    payload = json.loads(raw_out)
                except json.JSONDecodeError:
                    payload = None
        if not isinstance(payload, dict) or payload.get("error"):
            continue
        if tc.get("tool") == "aggregate" and payload.get("type") and "answer" in payload:
            verified.append({
                "index": idx,
                "messageID": tc.get("messageID"),
                "source": "tool:aggregate",
                "type": str(payload["type"]).strip().lower(),
                "answer": payload["answer"],
            })
        elif tc.get("tool") == "run_python" and payload.get("success") is True:
            po = payload.get("parsed_output")
            if isinstance(po, dict) and po.get("kind") == "final" and po.get("type"):
                try:
                    _, v_po = validate_protocol_payload(dict(po))
                    if _looks_like_wrapped_payload(v_po["type"], v_po.get("answer")):
                        # final sai format là dấu hiệu đáp án mới nhất không đáng tin: không lùi về final cũ của câu khác nghĩa
                        verified.clear()
                        continue
                    verified.append({
                        "index": idx,
                        "messageID": tc.get("messageID"),
                        "source": "tool:run_python",
                        "type": _coerce_month_string(str(v_po["type"]).strip().lower(), v_po.get("answer"))[0],
                        "answer": _coerce_month_string(str(v_po["type"]).strip().lower(), v_po.get("answer"))[1],
                    })
                except Exception:
                    pass
    return verified


def normalize_final_answer(question_id: str, raw_final_text: str, tool_calls: list) -> dict:
    """Chuẩn hóa final answer đúng Mục 4, xử lý xung đột kiểu giữa các tool theo thứ tự phép tính cuối."""
    verified_results = _extract_verified_tool_results_chronological(tool_calls)
    has_valid_tool_final = bool(verified_results)

    rescue_pred = None
    if verified_results:
        last_verified = verified_results[-1]
        rescue_pred = SubmissionItem(id=question_id, type=last_verified["type"], answer=last_verified["answer"]).to_dict()

    stripped = (raw_final_text or "").strip()
    non_think = _strip_thinking_blocks(stripped)
    if not stripped or not non_think:
        if verified_results and last_verified["source"] == "tool:run_python":
            return {
                "prediction": rescue_pred,
                "type_source": "tool:run_python",
                "abstain_reason": None,
                "tool_succeeded_but_final_empty": True,
                "tool_rescue_prediction": rescue_pred,
            }
        return {
            "prediction": SubmissionItem.abstain(question_id, reason="empty_final").to_dict(),
            "type_source": "abstain",
            "abstain_reason": "empty_final",
            "tool_succeeded_but_final_empty": has_valid_tool_final,
            "tool_rescue_prediction": rescue_pred,
        }

    low_nt = non_think.lower()
    step_limit_banner = (
        low_nt.startswith("do not invoke any tools.")
        or "the maximum number of steps has been reached" in low_nt
        or "the maximum number of steps permitted" in low_nt
        or "đã đạt số bước cực đại" in low_nt
    )

    bare_val = None
    forced_bare_from_tool = False
    if step_limit_banner:
        if not verified_results:
            return {
                "prediction": SubmissionItem.abstain(question_id, reason="step_limit_reached").to_dict(),
                "type_source": "abstain",
                "abstain_reason": "step_limit_reached",
                "tool_succeeded_but_final_empty": False,
                "tool_rescue_prediction": rescue_pred,
            }
        ok_scalar, scalar_val = _extract_verified_scalar_from_prose(stripped, verified_results, max_len=4000)
        bare_val = scalar_val if ok_scalar else verified_results[-1]["answer"]
        forced_bare_from_tool = True

    if not forced_bare_from_tool:
        candidates = _try_parse_json_candidates(stripped)
        if not candidates:
            ok_scalar, scalar_val = _extract_verified_scalar_from_prose(stripped, verified_results)
            if ok_scalar:
                candidates = [scalar_val]
        if not candidates:
            if rescue_pred is not None and _rescue_consistent_with_prose(stripped, rescue_pred):
                return {
                    "prediction": rescue_pred,
                    "type_source": "tool:run_python",
                    "abstain_reason": None,
                    "tool_succeeded_but_final_empty": False,
                    "tool_rescue_prediction": rescue_pred,
                }
            return {
                "prediction": SubmissionItem.abstain(question_id, reason="unparseable_final").to_dict(),
                "type_source": "abstain",
                "abstain_reason": "unparseable_final",
                "tool_succeeded_but_final_empty": False,
                "tool_rescue_prediction": rescue_pred,
            }

        for cand in candidates:
            if isinstance(cand, dict) and "type" in cand and cand["type"] is not None:
                ok, norm_ans, err = _validate_prediction_schema(cand["type"], cand.get("answer"))
                if ok:
                    pred = SubmissionItem(id=question_id, type=str(cand["type"]).strip().lower(), answer=norm_ans).to_dict()
                    return {
                        "prediction": pred,
                        "type_source": "model",
                        "abstain_reason": None,
                        "tool_succeeded_but_final_empty": False,
                        "tool_rescue_prediction": rescue_pred,
                    }
                return {
                    "prediction": SubmissionItem.abstain(question_id, reason=err).to_dict(),
                    "type_source": "abstain",
                    "abstain_reason": err,
                    "tool_succeeded_but_final_empty": False,
                    "tool_rescue_prediction": rescue_pred,
                }

        for cand in candidates:
            if isinstance(cand, dict) and "answer" in cand:
                bare_val = cand["answer"]
                break
            if isinstance(cand, dict) and "columns" in cand and "rows" in cand:
                bare_val = cand
                break
            if not isinstance(cand, dict):
                bare_val = cand
                break

    matching = [r for r in verified_results if _values_match(bare_val, r["answer"])]
    if not matching:
        return {
            "prediction": SubmissionItem.abstain(question_id, reason="ambiguous_type_without_matching_tool").to_dict(),
            "type_source": "abstain",
            "abstain_reason": "ambiguous_type_without_matching_tool",
            "tool_succeeded_but_final_empty": False,
            "tool_rescue_prediction": rescue_pred,
        }

    matched_types = {r["type"] for r in matching}
    chosen = matching[-1]
    if len(matched_types) > 1:
        # Nếu nhiều tool trả cùng giá trị nhưng khác kiểu, chỉ nhận kiểu của phép tính cuối thực sự
        is_overall_last = chosen["index"] == verified_results[-1]["index"]
        same_turn_conflict = (
            chosen["messageID"] is not None
            and any(r["type"] != chosen["type"] and r["messageID"] == chosen["messageID"] for r in matching)
        )
        if not is_overall_last or same_turn_conflict:
            return {
                "prediction": SubmissionItem.abstain(question_id, reason="conflicting_tool_types").to_dict(),
                "type_source": "abstain",
                "abstain_reason": "conflicting_tool_types",
                "tool_succeeded_but_final_empty": False,
                "tool_rescue_prediction": rescue_pred,
            }

    ok, norm_ans, err = _validate_prediction_schema(chosen["type"], bare_val)
    if ok:
        pred = SubmissionItem(id=question_id, type=chosen["type"], answer=norm_ans).to_dict()
        return {
            "prediction": pred,
            "type_source": chosen["source"],
            "abstain_reason": None,
            "tool_succeeded_but_final_empty": False,
            "tool_rescue_prediction": rescue_pred,
        }
    return {
        "prediction": SubmissionItem.abstain(question_id, reason=err or "ambiguous_type_without_matching_tool").to_dict(),
        "type_source": "abstain",
        "abstain_reason": err or "ambiguous_type_without_matching_tool",
        "tool_succeeded_but_final_empty": False,
        "tool_rescue_prediction": rescue_pred,
    }


_CATALOG_CACHE: dict = {}
_CATALOG_LOCK = threading.Lock()


def build_data_catalog(shop_root: Path, env: dict, cache_key: str, max_chars: int = 4500) -> str:
    """Quét cấu trúc file của shop bằng chính tool inspect_table (không dùng model) và gom thành mục lục ngắn.
    Chỉ mô tả file/sheet/cột; không chứa quy tắc nghiệp vụ. Lỗi ở bất kỳ bước nào thì bỏ qua phần đó."""
    with _CATALOG_LOCK:
        if cache_key in _CATALOG_CACHE:
            return _CATALOG_CACHE[cache_key]
    lines, other = [], []
    files = sorted(p for p in shop_root.iterdir() if p.is_file() and p.name.lower() != "readme.md")
    for p in files:
        if p.suffix.lower() not in {".csv", ".xlsx", ".xls"} or len(lines) >= 12:
            other.append(p.name)
            continue
        try:
            proc = subprocess.run(
                [sys.executable, str(OPENCODE_RUNTIME_DIR / "bridge.py"), "inspect_table", json.dumps({"file": p.name, "column": None}, ensure_ascii=False)],
                env=env, capture_output=True, text=True, timeout=45, check=True,
            )
            info = json.loads(proc.stdout.strip())
        except Exception:
            other.append(p.name)
            continue
        previews = info.get("sheets_preview")
        if isinstance(previews, dict) and previews:
            groups: dict = {}
            for sname, sp in previews.items():
                sp = sp or {}
                cols = [str(c) for c in (sp.get("columns") or [])]
                hdr = sp.get("header_row_index")
                rows = [r for r in (sp.get("first_rows") or []) if isinstance(r, list)]
                if rows:
                    # header đoán của bridge có thể rơi vào khối tiêu đề sổ: lấy dòng đầu tiên có nhiều ô chữ nhất trong các dòng đầu
                    cnt = [len([c for c in r if str(c).strip()]) for r in rows]
                    k = cnt.index(max(cnt))
                    cols, hdr = [str(c) for c in rows[k] if str(c).strip()], (hdr if k == hdr else None)
                cols = list(dict.fromkeys(cols))[:24]
                groups.setdefault((hdr, tuple(cols)), []).append(sname)
            for (hdr, cols), names in groups.items():
                shown = ", ".join(f'"{n}"' for n in names[:4]) + (f" (+{len(names) - 4} sheet cùng cột)" if len(names) > 4 else "")
                lines.append(f"- {p.name} | sheet {shown}" + (f" | dòng tiêu đề {hdr + 1}" if isinstance(hdr, int) else "") + (f" | cột (đoán từ vài dòng đầu): {', '.join(cols)}" if cols else ""))
        else:
            cols = [str(c) for c in (info.get("columns") or [])][:30]
            lines.append(f"- {p.name} | cột (đoán từ vài dòng đầu): {', '.join(cols)}" if cols else f"- {p.name}")
    if other:
        lines.append("- file khác (xem bằng inspect_table / run_python): " + ", ".join(other[:15]) + (f" (+{len(other) - 15})" if len(other) > 15 else ""))
    text = ""
    if lines:
        text = "\n".join(lines)
        if len(text) > max_chars:
            text = text[:max_chars].rsplit("\n", 1)[0] + "\n- ... (còn nữa, dùng inspect_table để xem đầy đủ)"
        text = ("## Mục lục dữ liệu của shop (tự động quét cấu trúc; chỉ để định hướng, KHÔNG phải quy tắc tính; "
                "quy tắc và cách ghi dữ liệu vẫn lấy từ rulebook và README)\n" + text)
    with _CATALOG_LOCK:
        _CATALOG_CACHE[cache_key] = text
    return text


def run_single_question(
    question_item: QuestionItem,
    split_dir: Path,
    run_out_dir: Path,
    tool_mode: str,
    temperature: float,
    ollama_tag: str,
    ollama_port: int = 11434,
    worker_id: int = 0,
) -> dict:
    """Chạy 1 câu hỏi end-to-end qua OpenCode CLI + Ollama loopback và lưu trace đầy đủ."""
    t0 = time.time()
    q_root_base = run_out_dir / "_shop_roots"
    shop_root = prepare_isolated_shop_root(question_item, split_dir, q_root_base)
    session_dir = setup_opencode_session_dir(run_out_dir / "_opencode_sessions" / f"{question_item.id}_w{worker_id}", tool_mode)
    work_dir = run_out_dir / "_python_workdir" / question_item.id
    work_dir.mkdir(parents=True, exist_ok=True)
    sidecar_path = run_out_dir / "_tool_sidecar" / f"{question_item.id}.jsonl"
    if sidecar_path.exists():
        sidecar_path.unlink()

    sys_instr_path = session_dir / "datamate_instructions.md"
    sys_instr_text = build_system_instructions(shop_root, tool_mode)
    sys_instr_path.write_text(sys_instr_text, encoding="utf-8")

    permissions = {"*": "deny", "read": "allow", "inspect_table": "allow", "aggregate": "allow"}
    if tool_mode == "eval_agent":
        permissions["run_python"] = "allow"
        permissions["explore_data"] = "allow"
    if tool_mode == "auto_readme_schema":
        permissions.pop("read", None)

    effective_max_steps = MAX_STEPS_EVAL_AGENT if tool_mode == "eval_agent" else MAX_STEPS
    effective_tool_max_bytes = TOOL_MAX_BYTES_EVAL_AGENT if tool_mode == "eval_agent" else TOOL_MAX_BYTES

    instructions_order = (
        [str(RULEBOOK_PATH), str(sys_instr_path)]
        if tool_mode == "eval_agent"
        else [str(sys_instr_path), str(RULEBOOK_PATH)]
    )

    opencode_config = {
        "enabled_providers": ["ollama-local"],
        "provider": {
            "ollama-local": {
                "name": "Local Ollama",
                "npm": "@ai-sdk/openai-compatible",
                "options": {"baseURL": f"http://127.0.0.1:{ollama_port}/v1"},
                "models": {
                    ollama_tag: {
                        "name": BASE_MODEL_TAG,
                        "tool_call": True,
                        "limit": {"context": NUM_CTX, "output": MAX_OUTPUT_TOKENS},
                    }
                },
            }
        },
        "model": f"ollama-local/{ollama_tag}",
        "small_model": f"ollama-local/{ollama_tag}",
        "instructions": instructions_order,
        "autoupdate": False,
        "share": "disabled",
        "compaction": {"auto": False},
        "tool_output": {"max_bytes": effective_tool_max_bytes},
        "permission": permissions,
        "agent": {"build": {"temperature": temperature, "steps": effective_max_steps}},
    }

    user_prompt = build_user_prompt(question_item, shop_root, tool_mode)
    env = os.environ.copy()
    for _tok_key in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HF_HUB_TOKEN"):
        env.pop(_tok_key, None)
    env.update({
        "HOME": str(session_dir),
        "OPENCODE_CONFIG_DIR": str(session_dir),
        "OPENCODE_CONFIG_CONTENT": json.dumps(opencode_config, ensure_ascii=False),
        "OPENCODE_DISABLE_MODELS_FETCH": "1",
        "OPENCODE_DISABLE_AUTOUPDATE": "1",
        "DATAMATE_PYTHON_BIN": sys.executable,
        "DATAMATE_BRIDGE_PATH": str(OPENCODE_RUNTIME_DIR / "bridge.py"),
        "DATAMATE_SRC_ROOT": str(STRUCTURED_SRC_ROOT),
        "DATAMATE_DATA_ROOT": str(shop_root),
        "DATAMATE_WORK_DIR": str(work_dir),
        "DATAMATE_AS_OF_DATE": question_item.as_of_date,
        "DATAMATE_RULEBOOK_PATH": str(RULEBOOK_PATH),
        "DATAMATE_CALENDAR_PATH": str(CALENDAR_PATH) if CALENDAR_PATH else "",
        "DATAMATE_TIMEOUT_SEC": str(RUN_PYTHON_TIMEOUT_SEC),
        "DATAMATE_TOOL_TIMEOUT_MS": str(OPENCODE_TOOL_TIMEOUT_MS),
        "DATAMATE_TOOL_TRACE_PATH": str(sidecar_path),
        "DATAMATE_AUTO_WRAP_STDOUT": "1",
    })

    if os.environ.get("DATAMATE_CATALOG", "1") != "0":
        try:
            _cat = build_data_catalog(shop_root, env, f"{question_item.business_id}")
            if _cat:
                sys_instr_text += "\n\n" + _cat
                sys_instr_path.write_text(sys_instr_text, encoding="utf-8")
        except Exception:
            pass   # mục lục chỉ là gợi ý: lỗi thì bỏ qua, không làm hỏng câu hỏi

    cmd = [OPENCODE_BIN, "--pure", "-m", f"ollama-local/{ollama_tag}", "run", "--format", "json"]

    def run_turn(args: list[str], turn_env: dict, timeout: float):
        try:
            proc = subprocess.run(
                cmd + args, cwd=str(shop_root), env=turn_env, capture_output=True,
                text=True, encoding="utf-8", errors="replace", timeout=timeout,
            )
            return proc.stdout or "", proc.stderr or "", proc.returncode, False
        except subprocess.TimeoutExpired as exc:
            stdout = exc.stdout.decode("utf-8", errors="replace") if isinstance(exc.stdout, bytes) else exc.stdout
            stderr = exc.stderr.decode("utf-8", errors="replace") if isinstance(exc.stderr, bytes) else exc.stderr
            return stdout or "", stderr or f"Timeout after {timeout:.0f}s", -1, True

    def parse_turns(stdout: str):
        sidecar_traces = []
        if sidecar_path.is_file():
            for line in sidecar_path.read_text(encoding="utf-8").splitlines():
                if line.strip():
                    try:
                        sidecar_traces.append(json.loads(line))
                    except json.JSONDecodeError:
                        pass
        return parse_opencode_jsonl_events(stdout, sidecar_traces=sidecar_traces)

    stdout_text, stderr_text, exit_code, timed_out = run_turn([user_prompt], env, PER_QUESTION_TIMEOUT_SEC)
    parsed = parse_turns(stdout_text)
    norm = normalize_final_answer(
        question_item.id, "" if timed_out else parsed["raw_final_text"], parsed["tool_calls"]
    )
    recovery_attempted = False
    time_left = PER_QUESTION_TIMEOUT_SEC - (time.time() - t0)
    _ran_python = any(tc["tool"] == "run_python" and tc["status"] == "completed" for tc in parsed["tool_calls"])
    # Lượt cuối bị cắt khi chưa chốt (suy luận dài hết token, nội dung rỗng): cho chạy tiếp cùng phiên, buộc ra bước tool kế tiếp
    _truncated = norm["abstain_reason"] == "empty_final" and not _ran_python and os.environ.get("DATAMATE_TRUNC_RECOVERY", "1") != "0"
    if (
        tool_mode == "eval_agent" and exit_code == 0 and parsed["session_id"] and time_left > 0
        and norm["abstain_reason"] and norm["tool_rescue_prediction"] is None
        and (_ran_python or _truncated)
    ):
        recovery_attempted = True
        if _ran_python:
            opencode_config["permission"] = {"*": "deny", "run_python": "allow"}
            opencode_config["agent"]["build"]["steps"] = 3
        else:
            opencode_config["permission"] = {**permissions, "*": "deny"}
            opencode_config["agent"]["build"]["steps"] = 8
        retry_env = {**env, "OPENCODE_CONFIG_CONTENT": json.dumps(opencode_config, ensure_ascii=False)}
        retry_prompt = (
            "Lượt trước bị cắt vì suy luận quá dài khi chưa có kết quả. KHÔNG suy luận dài thêm (tối đa vài câu mỗi lượt): "
            "chưa đủ dữ kiện thì gọi ngay một tool để lấy dữ kiện, đủ rồi thì gọi run_python tính và emit_final. "
            "Câu hỏi và rulebook vẫn như ban đầu; chỉ chốt khi dữ kiện đủ."
        ) if _truncated else (
            "Chốt câu hỏi ban đầu trong phiên này. Dựa vào file, sheet, cột và điều kiện đã xác minh, "
            "gọi ngay run_python để đọc dữ liệu gốc và tính đúng giá trị cần trả. "
            "Các helper emit_obs, emit_final, load_clean_df, explore_data, parse_vn_number, resolve_date đã có sẵn trong globals (KHÔNG import). Sau khi emit_final thành công, xuất câu trả lời cuối cùng chứa đáp án. "
            "Dòng cuối dùng emit_final(answer, type=..., evidence=...) với biến answer vừa tính. "
            "Giữ mọi xử lý trong bộ nhớ. Chỉ chốt khi có dữ kiện đủ để tính."
        )
        retry_stdout, retry_stderr, exit_code, timed_out = run_turn(
            ["--session", parsed["session_id"], retry_prompt], retry_env, min(time_left, 120 if _ran_python else 240)
        )
        parsed = parse_turns(stdout_text + "\n" + retry_stdout)
        stderr_text = "\n".join(s for s in (stderr_text, retry_stderr) if s)
        recovered = normalize_final_answer(question_item.id, "", parsed["tool_calls"])
        if recovered["type_source"] == "tool:run_python":
            recovered["tool_succeeded_but_final_empty"] = not _strip_thinking_blocks(parsed["raw_final_text"])
            norm = recovered

    if timed_out and norm["type_source"] != "tool:run_python":
        norm["abstain_reason"] = "question_timeout"
        norm["tool_succeeded_but_final_empty"] = False
    elapsed = round(time.time() - t0, 3)

    return {
        "id": question_item.id,
        "business_id": question_item.business_id,
        "as_of_date": question_item.as_of_date,
        "tool_mode": tool_mode,
        "temperature": temperature,
        "worker_id": worker_id,
        "ollama_port": ollama_port,
        "user_prompt": user_prompt,
        "system_instructions": sys_instr_text,
        "rulebook_path": str(RULEBOOK_PATH),
        "rulebook_sha256": RULEBOOK_SHA256,
        "shop_root": str(shop_root),
        "exit_code": exit_code,
        "timed_out": timed_out,
        "finalization_retry_attempted": recovery_attempted,
        "elapsed_sec": elapsed,
        "finish_reason": parsed["finish_reason"],
        "final_message_id": parsed["final_message_id"],
        "tokens": parsed["token_summary"],
        "tool_calls": parsed["tool_calls"],
        "api_errors": parsed["api_errors"],
        "jsonl_parse_errors": parsed["jsonl_parse_errors"],
        "raw_final_text": parsed["raw_final_text"],
        "prediction": norm["prediction"],
        "type_source": norm["type_source"],
        "abstain_reason": norm["abstain_reason"],
        "tool_succeeded_but_final_empty": norm["tool_succeeded_but_final_empty"],
        "tool_rescue_prediction": norm["tool_rescue_prediction"],
        "stderr": stderr_text[-2000:] if stderr_text else "",
    }


ALL_QUESTIONS = {}
if SPLIT_DIR and (SPLIT_DIR / "questions.jsonl").is_file():
    with (SPLIT_DIR / "questions.jsonl").open("r", encoding="utf-8-sig") as fh:
        for line in fh:
            if line.strip():
                qi = QuestionItem.from_dict(json.loads(line))
                ALL_QUESTIONS[qi.id] = qi
    print(f"Loaded {len(ALL_QUESTIONS)} questions from {SPLIT_DIR / 'questions.jsonl'}.")
else:
    print("SPLIT_DIR chưa nạp questions.jsonl (chế độ score_only).")


In [ ]:
# ==============================================================================
# CELL 4: SMOKE TEST TRÊN 1× T4 & GATE
# ==============================================================================

def log_gpu_and_ollama_ps(port: int = 11434, label: str = ""):
    print(f"--- GPU & ollama ps ({label}) ---")
    ps = subprocess.run([OLLAMA_BIN, "ps"], env={**os.environ, "OLLAMA_HOST": f"127.0.0.1:{port}"}, capture_output=True, text=True, check=False)
    print(ps.stdout.strip() or ps.stderr.strip())
    smi = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total,utilization.gpu", "--format=csv,noheader"], capture_output=True, text=True, check=False)
    print(smi.stdout.strip())


def diagnose_smoke_stage(trace: dict, expected_type: str, expected_answer) -> tuple[bool, str]:
    tools_used = [tc.get("tool") for tc in trace.get("tool_calls", [])]
    if not tools_used:
        return False, "Lệch ở khâu khám phá dữ liệu: model chưa gọi tool nào."
    if not _extract_verified_tool_results_chronological(trace.get("tool_calls", [])):
        return False, "Lệch ở khâu chọn tham số / thực thi tool (aggregate hoặc run_python chưa ra kết quả hợp lệ)."
    pred = trace.get("prediction", {})
    if pred.get("type") == "abstain":
        return False, f"Tool tính xong nhưng lệch ở khâu chốt đáp án (abstain_reason={trace.get('abstain_reason')})."
    if pred.get("type") == expected_type and _values_match(pred.get("answer"), expected_answer):
        return True, "PASS: Khám phá -> tham số -> thực thi -> chốt đáp án đều khớp."
    return False, f"Lệch giá trị chốt đáp án: got={pred}, expected={{'type': {expected_type!r}, 'answer': {expected_answer!r}}}."


def print_smoke_failure_details(label: str, trace: dict, trace_path: Path) -> None:
    print(f"=== DIAGNOSTIC CHI TIẾT ({label}) ===")
    print(f"trace_path        : {trace_path}")
    print(f"exit_code/timeout : {trace.get('exit_code')} / timed_out={trace.get('timed_out')}")
    print(f"finish_reason     : {trace.get('finish_reason')} | tokens={trace.get('tokens')}")
    print(f"raw_final_text    : {repr((trace.get('raw_final_text') or '')[:1200])}")
    print(f"tool_rescue_pred  : {trace.get('tool_rescue_prediction')}")
    print(f"finalization_retry: {trace.get('finalization_retry_attempted', False)}")
    if trace.get("api_errors"):
        print(f"api_errors        : {trace.get('api_errors')}")
    if trace.get("jsonl_parse_errors"):
        print(f"jsonl_parse_errors: {trace.get('jsonl_parse_errors')[:3]}")
    for idx, tc in enumerate(trace.get("tool_calls", []), 1):
        res_preview = tc.get("full_result") if tc.get("full_result") is not None else tc.get("output")
        print(f"  tool[{idx}] {tc.get('tool')} ({tc.get('status')}) input={tc.get('input')} -> result={repr(res_preview)[:500]}")
    if trace.get("stderr"):
        print(f"stderr_tail       : {repr(trace.get('stderr')[-500:])}")


SMOKE_RESULTS = {}
SMOKE_PASSED = False

if EXECUTION_STAGE == "inference" and RUN_SMOKE_TESTS:
    smoke_dir = RUN_DIR / "smoke"
    smoke_dir.mkdir(parents=True, exist_ok=True)
    log_gpu_and_ollama_ps(11434, "after 16K model load on 1x T4")

    # Smoke 1: kiểm tra eval_agent (run_python + multi-sheet/mojibake XLSX)
    q1 = ALL_QUESTIONS[SMOKE_RUN_PYTHON_QID]
    t1 = run_single_question(
        q1, SPLIT_DIR, smoke_dir / "smoke_run_python",
        tool_mode="eval_agent", temperature=0.0, ollama_tag=SMOKE_TAG, ollama_port=11434, worker_id=0,
    )
    smoke1_path = smoke_dir / f"{SMOKE_RUN_PYTHON_QID}.json"
    smoke1_path.write_text(json.dumps(t1, indent=2, ensure_ascii=False), encoding="utf-8")
    used_run_python = any(tc.get("tool") == "run_python" for tc in t1.get("tool_calls", []))
    ok1_raw, diag1 = diagnose_smoke_stage(t1, *SMOKE_RUN_PYTHON_EXPECTED)
    ok1 = ok1_raw and used_run_python
    if ok1_raw and not used_run_python:
        diag1 = "Chưa thấy model gọi run_python trong trace."
    print(f"[SMOKE 1 - eval_agent] {SMOKE_RUN_PYTHON_QID} | used_run_python={used_run_python} | pred={t1['prediction']} | type_source={t1['type_source']} | {t1['elapsed_sec']}s | {diag1}")
    if not ok1:
        print_smoke_failure_details("SMOKE 1", t1, smoke1_path)
    SMOKE_RESULTS["smoke_run_python"] = {"passed": ok1, "trace": t1, "used_run_python": used_run_python, "diagnostic": diag1}

    # Smoke 2: dùng aggregate (structured_only) để đếm đơn và kiểm tra runtime
    ok2 = True
    run_smoke_2 = (not SMOKE_SINGLE_QUESTION_ONLY) or (RUN_BATCH_EVAL and TOOL_MODE == "structured_only")
    if run_smoke_2:
        q2 = ALL_QUESTIONS[SMOKE_AGGREGATE_QID]
        t2 = run_single_question(
            q2, SPLIT_DIR, smoke_dir / "smoke_agg",
            tool_mode="structured_only", temperature=0.0, ollama_tag=SMOKE_TAG, ollama_port=11434, worker_id=0,
        )
        smoke2_path = smoke_dir / f"{SMOKE_AGGREGATE_QID}.json"
        smoke2_path.write_text(json.dumps(t2, indent=2, ensure_ascii=False), encoding="utf-8")
        ok2, diag2 = diagnose_smoke_stage(t2, "number", 41)
        print(f"[SMOKE 2 - structured_only] {SMOKE_AGGREGATE_QID} | pred={t2['prediction']} | type_source={t2['type_source']} | {t2['elapsed_sec']}s | {diag2}")
        if not ok2:
            print_smoke_failure_details("SMOKE 2", t2, smoke2_path)
        SMOKE_RESULTS["smoke_aggregate"] = {"passed": ok2, "trace": t2, "diagnostic": diag2}

    SMOKE_PASSED = bool(ok1 and ok2)
    if not SMOKE_PASSED and FAIL_FAST_ON_SMOKE_ERROR:
        raise RuntimeError(
            "Smoke test thất bại — dừng sớm trước khi chạy batch eval. "
            f"Smoke 1: {diag1}" + (f" | Smoke 2: {SMOKE_RESULTS.get('smoke_aggregate', {}).get('diagnostic')}" if run_smoke_2 else "")
        )

    if ENABLE_DUAL_WORKER_BENCHMARK and GPU_COUNT >= 2:
        print("\nĐang kiểm tra dual-worker song song trên GPU 0 (11434) và GPU 1 (11435)...")
        if len(OLLAMA_WORKERS) < 2:
            w1 = start_ollama_worker(1, 11435, cuda_visible_devices="1")
            OLLAMA_WORKERS.append(w1)
            ensure_base_model_on_worker(11435)
        create_16k_tag(11435, SMOKE_TAG, temperature=0.0)
        warm1 = urllib.request.Request(
            "http://127.0.0.1:11435/api/generate",
            data=json.dumps({"model": SMOKE_TAG, "prompt": "OK", "stream": False, "options": {"num_predict": 1}}).encode("utf-8"),
            headers={"Content-Type": "application/json"},
        )
        with urllib.request.urlopen(warm1, timeout=180) as _resp:
            _ = json.load(_resp)
        q_w0 = ALL_QUESTIONS[SMOKE_AGGREGATE_QID]
        q_w1 = ALL_QUESTIONS[SMOKE_RUN_PYTHON_QID]
        dual_t0 = time.time()
        with ThreadPoolExecutor(max_workers=2) as pool:
            f0 = pool.submit(run_single_question, q_w0, SPLIT_DIR, smoke_dir / "dual_w0", "eval_agent", 0.0, SMOKE_TAG, 11434, 0)
            f1 = pool.submit(run_single_question, q_w1, SPLIT_DIR, smoke_dir / "dual_w1", "eval_agent", 0.0, SMOKE_TAG, 11435, 1)
            r0, r1 = f0.result(), f1.result()
        dual_w0_path = smoke_dir / "dual_w0.json"
        dual_w1_path = smoke_dir / "dual_w1.json"
        dual_w0_path.write_text(json.dumps(r0, indent=2, ensure_ascii=False), encoding="utf-8")
        dual_w1_path.write_text(json.dumps(r1, indent=2, ensure_ascii=False), encoding="utf-8")
        ok_dw0, diag_dw0 = diagnose_smoke_stage(r0, "number", 41)
        ok_dw1, diag_dw1 = diagnose_smoke_stage(r1, *SMOKE_RUN_PYTHON_EXPECTED)
        dual_elapsed = round(time.time() - dual_t0, 2)
        print(
            f"Dual-worker 2 câu xong trong {dual_elapsed}s "
            f"(w0={r0['elapsed_sec']}s pass={ok_dw0} pred={r0['prediction']}, "
            f"w1={r1['elapsed_sec']}s pass={ok_dw1} pred={r1['prediction']})."
        )
        if not ok_dw0:
            print(f"[DUAL_W0 DIAG] {diag_dw0}")
            print_smoke_failure_details("DUAL_W0", r0, dual_w0_path)
        if not ok_dw1:
            print(f"[DUAL_W1 DIAG] {diag_dw1}")
            print_smoke_failure_details("DUAL_W1", r1, dual_w1_path)
        log_gpu_and_ollama_ps(11435, "after dual-worker on GPU 0 + GPU 1")
        SMOKE_RESULTS["dual_worker_probe_sec"] = dual_elapsed
        SMOKE_RESULTS["dual_w0_passed"] = ok_dw0
        SMOKE_RESULTS["dual_w1_passed"] = ok_dw1
elif EXECUTION_STAGE == "score_only":
    print("EXECUTION_STAGE='score_only' -> bỏ qua smoke inference.")


In [ ]:
# ==============================================================================
# CELL 5: EVAL THEO MANIFEST 50 CÂU (+ ĐỐI CHỨNG CSV COUNT_DISTINCT)
# ==============================================================================
import queue
import threading


def ensure_workers_started(desired_workers: int) -> list[dict]:
    if desired_workers >= 2 and GPU_COUNT >= 2 and len(OLLAMA_WORKERS) < 2:
        w1 = start_ollama_worker(1, 11435, cuda_visible_devices="1")
        OLLAMA_WORKERS.append(w1)
        ensure_base_model_on_worker(11435)
    return OLLAMA_WORKERS[: max(1, min(desired_workers, len(OLLAMA_WORKERS)))]


def _warmup_worker_tag(port: int, tag_name: str) -> None:
    req = urllib.request.Request(
        f"http://127.0.0.1:{port}/api/generate",
        data=json.dumps({"model": tag_name, "prompt": "OK", "stream": False, "options": {"num_predict": 1}}).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=180) as resp:
        _ = json.load(resp)


def run_manifest_eval(
    run_label: str,
    question_ids: list[str],
    tool_mode: str,
    temperature: float,
    num_workers: int = 1,
) -> Path:
    sub_dir = RUN_DIR / run_label
    traces_dir = sub_dir / "traces"
    sub_dir.mkdir(parents=True, exist_ok=True)
    traces_dir.mkdir(parents=True, exist_ok=True)

    shutil.copy2(RULEBOOK_PATH, sub_dir / "rulebook_snapshot.md")

    active_daemons = ensure_workers_started(num_workers)
    total_slots = max(1, min(num_workers, len(active_daemons) * PARALLEL_SLOTS_PER_GPU))
    active_slots = [
        {
            "slot_id": slot_idx,
            "worker_idx": active_daemons[slot_idx % len(active_daemons)]["worker_idx"],
            "port": active_daemons[slot_idx % len(active_daemons)]["port"],
        }
        for slot_idx in range(total_slots)
    ]

    tag_name = f"datamate-16k-{run_label}-{RUN_TIMESTAMP}:latest".replace("_", "-").lower()
    for w in active_daemons:
        create_16k_tag(w["port"], tag_name, temperature=temperature)
    with ThreadPoolExecutor(max_workers=len(active_daemons)) as warm_pool:
        list(warm_pool.map(lambda w: _warmup_worker_tag(w["port"], tag_name), active_daemons))
    log_gpu_and_ollama_ps(
        active_daemons[-1]["port"],
        f"batch warmup {len(active_daemons)} GPU daemons × {PARALLEL_SLOTS_PER_GPU} slots = {len(active_slots)} workers ({run_label})",
    )

    manifest = {
        "run_label": run_label,
        "split": SPLIT,
        "tool_mode": tool_mode,
        "temperature": temperature,
        "seed": SEED,
        "num_ctx": NUM_CTX,
        "ollama_kv_ctx": NUM_CTX,
        "num_batch": NUM_BATCH,
        "presence_penalty": PRESENCE_PENALTY,
        "max_output_tokens": MAX_OUTPUT_TOKENS,
        "max_steps": MAX_STEPS_EVAL_AGENT if tool_mode == "eval_agent" else MAX_STEPS,
        "tool_max_bytes": TOOL_MAX_BYTES_EVAL_AGENT if tool_mode == "eval_agent" else TOOL_MAX_BYTES,
        "run_python_timeout_sec": RUN_PYTHON_TIMEOUT_SEC,
        "base_model_tag": BASE_MODEL_TAG,
        "expected_model_digest": EXPECTED_MODEL_DIGEST,
        "expected_snapshot_commit": EXPECTED_SNAPSHOT_COMMIT,
        "snapshot_src_source": SNAPSHOT_SRC_SOURCE,
        "rulebook_path": str(RULEBOOK_PATH),
        "rulebook_sha256": RULEBOOK_SHA256,
        "question_count": len(question_ids),
        "question_ids": list(question_ids),
        "parallel_slots_per_gpu": PARALLEL_SLOTS_PER_GPU,
        "num_workers": len(active_slots),
        "worker_ports": [s["port"] for s in active_slots],
        "runtime_info": RUNTIME_INFO,
        "scoring_policy": "baseline_official_with_handoff_section4_type_inference",
    }
    (sub_dir / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

    pred_path = sub_dir / "predictions.jsonl"
    if pred_path.exists():
        pred_path.unlink()

    valid_q_items = [ALL_QUESTIONS[qid] for qid in question_ids if qid in ALL_QUESTIONS]
    missing_qids = [qid for qid in question_ids if qid not in ALL_QUESTIONS]
    if missing_qids:
        print(f"[WARN] {len(missing_qids)} ID không có trong {SPLIT}: {missing_qids[:5]}")

    batch_start = time.time()
    results_by_idx = {}

    def _format_trace_log(tr: dict) -> str:
        extra = ""
        if tr["prediction"].get("type") in ("abstain", "unanswerable"):
            t_names = [tc.get("tool") for tc in tr.get("tool_calls", [])]
            extra = f" | reason={tr.get('abstain_reason')} | finish={tr.get('finish_reason')} | tools={t_names}"
        return f"{tr['prediction']} | src={tr['type_source']} | {tr['elapsed_sec']}s{extra}"

    if len(active_slots) == 1:
        slot = active_slots[0]
        for i, qi in enumerate(valid_q_items, 1):
            tr = run_single_question(
                qi, SPLIT_DIR, sub_dir,
                tool_mode=tool_mode,
                temperature=temperature,
                ollama_tag=tag_name,
                ollama_port=slot["port"],
                worker_id=slot["slot_id"],
            )
            results_by_idx[i - 1] = tr
            (traces_dir / f"{qi.id}.json").write_text(json.dumps(tr, indent=2, ensure_ascii=False), encoding="utf-8")
            with pred_path.open("a", encoding="utf-8") as pfh:
                pfh.write(json.dumps(tr["prediction"], ensure_ascii=False) + "\n")
            print(f"  [{run_label}] {i}/{len(valid_q_items)} [gpu{slot['worker_idx']}/s{slot['slot_id']}] {qi.id} ({qi.business_id}) -> {_format_trace_log(tr)}")
    else:
        work_q = queue.Queue()
        for idx, qi in enumerate(valid_q_items):
            work_q.put((idx, qi))
        io_lock = threading.Lock()
        done_count = 0

        def _worker_loop(slot: dict) -> None:
            nonlocal done_count
            while True:
                try:
                    idx, qi = work_q.get_nowait()
                except queue.Empty:
                    return
                try:
                    tr = run_single_question(
                        qi, SPLIT_DIR, sub_dir,
                        tool_mode=tool_mode,
                        temperature=temperature,
                        ollama_tag=tag_name,
                        ollama_port=slot["port"],
                        worker_id=slot["slot_id"],
                    )
                    with io_lock:
                        done_count += 1
                        results_by_idx[idx] = tr
                        (traces_dir / f"{tr['id']}.json").write_text(json.dumps(tr, indent=2, ensure_ascii=False), encoding="utf-8")
                        print(f"  [{run_label}] {done_count}/{len(valid_q_items)} [gpu{slot['worker_idx']}/s{slot['slot_id']}] {tr['id']} ({tr['business_id']}) -> {_format_trace_log(tr)}")
                finally:
                    work_q.task_done()

        with ThreadPoolExecutor(max_workers=len(active_slots)) as pool:
            futs = [pool.submit(_worker_loop, slot) for slot in active_slots]
            for fut in futs:
                fut.result()

        with pred_path.open("w", encoding="utf-8") as pfh:
            for idx in range(len(valid_q_items)):
                pfh.write(json.dumps(results_by_idx[idx]["prediction"], ensure_ascii=False) + "\n")

    manifest["total_elapsed_sec"] = round(time.time() - batch_start, 2)
    (sub_dir / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    return sub_dir


EVAL_SUBDIRS = []
if EXECUTION_STAGE == "inference" and RUN_BATCH_EVAL:
    if REQUIRE_SMOKE_PASS_FOR_BATCH and RUN_SMOKE_TESTS and not SMOKE_PASSED:
        raise RuntimeError("Batch eval bị chặn vì smoke test chưa pass.")
    for temp in TEMPERATURES:
        t_lbl = str(temp).replace(".", "")
        sub = run_manifest_eval(
            run_label=f"{TOOL_MODE}_50_t{t_lbl}",
            question_ids=MANIFEST_50_IDS,
            tool_mode=TOOL_MODE,
            temperature=temp,
            num_workers=NUM_WORKERS,
        )
        EVAL_SUBDIRS.append(sub)

    if RUN_CONTROL_CSV_COUNT_DISTINCT and SPLIT == "train":
        for temp in TEMPERATURES:
            t_lbl = str(temp).replace(".", "")
            sub = run_manifest_eval(
                run_label=f"control_csv_count_distinct_12_t{t_lbl}",
                question_ids=CSV_COUNT_DISTINCT_HOLDOUT_IDS,
                tool_mode="structured_only",
                temperature=temp,
                num_workers=NUM_WORKERS,
            )
            EVAL_SUBDIRS.append(sub)
elif EXECUTION_STAGE == "inference":
    print("RUN_BATCH_EVAL=False -> Chỉ mới chạy smoke test. Đổi RUN_BATCH_EVAL=True sau khi smoke pass.")


In [ ]:
# ==============================================================================
# CELL 6: CHẤM ĐIỂM PA (PHIÊN SCORE_ONLY) & XUẤT BÁO CÁO ĐỐI CHIẾU
# ==============================================================================
from collections import Counter


def get_shop_format_label(business_id: str) -> str:
    if not SPLIT_DIR:
        return "unknown"
    bdir = SPLIT_DIR / "data" / business_id
    if not bdir.is_dir():
        return "unknown"
    exts = sorted({p.suffix.lower().lstrip(".") for p in bdir.iterdir() if p.is_file() and p.suffix.lower() in {".csv", ".xlsx", ".xls", ".sqlite", ".db", ".sqlite3"}})
    return "+".join(exts) if exts else "unknown"


def score_run_dir(sub_dir: Path, ref_map: dict | None) -> dict:
    manifest = json.loads((sub_dir / "manifest.json").read_text(encoding="utf-8"))
    qids = manifest["question_ids"]
    preds = [json.loads(l) for l in (sub_dir / "predictions.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]

    traces = {}
    for qid in qids:
        tp = sub_dir / "traces" / f"{qid}.json"
        if tp.is_file():
            traces[qid] = json.loads(tp.read_text(encoding="utf-8"))

    type_sources = Counter(tr.get("type_source", "unknown") for tr in traces.values())
    abstain_reasons = Counter(tr.get("abstain_reason") for tr in traces.values() if tr.get("abstain_reason"))
    tool_empty_final_count = sum(1 for tr in traces.values() if tr.get("tool_succeeded_but_final_empty"))
    total_time = manifest.get("total_elapsed_sec") or round(sum(tr.get("elapsed_sec", 0.0) for tr in traces.values()), 2)

    result = {
        "run_label": manifest["run_label"],
        "tool_mode": manifest["tool_mode"],
        "temperature": manifest["temperature"],
        "question_count": len(qids),
        "total_elapsed_sec": total_time,
        "type_sources": dict(type_sources),
        "abstain_reasons": dict(abstain_reasons),
        "tool_succeeded_but_final_empty": tool_empty_final_count,
    }

    if ref_map is None:
        result["scoring_status"] = "PA_UNAVAILABLE_NO_REFERENCE"
        return result

    filtered_refs = [ref_map[qid] for qid in qids if qid in ref_map]
    if len(filtered_refs) != len(qids):
        result["scoring_status"] = f"PARTIAL_REFERENCE ({len(filtered_refs)}/{len(qids)})"
    else:
        result["scoring_status"] = "SCORED"

    report = evaluate_predictions(preds, filtered_refs)
    result.update({
        "scored_total": report.total,
        "correct": report.correct,
        "abstain": report.abstain,
        "wrong": report.wrong,
        "pa": round(report.penalized_accuracy, 6),
        "by_type": report.by_type,
    })

    rescue_preds = []
    for p in preds:
        tr = traces.get(p["id"], {})
        if tr.get("tool_succeeded_but_final_empty") and tr.get("tool_rescue_prediction"):
            rescue_preds.append(tr["tool_rescue_prediction"])
        else:
            rescue_preds.append(p)
    rescue_report = evaluate_predictions(rescue_preds, filtered_refs)
    result["shadow_tool_rescue"] = {
        "correct": rescue_report.correct,
        "abstain": rescue_report.abstain,
        "wrong": rescue_report.wrong,
        "pa": round(rescue_report.penalized_accuracy, 6),
    }

    by_format = {}
    for fmt_name in sorted({get_shop_format_label(ALL_QUESTIONS[qid].business_id) for qid in qids if qid in ALL_QUESTIONS}):
        fmt_ids = {qid for qid in qids if qid in ALL_QUESTIONS and get_shop_format_label(ALL_QUESTIONS[qid].business_id) == fmt_name}
        fmt_preds = [p for p in preds if p["id"] in fmt_ids]
        fmt_refs = [r for r in filtered_refs if r["id"] in fmt_ids]
        if fmt_refs:
            fr = evaluate_predictions(fmt_preds, fmt_refs)
            by_format[fmt_name] = {"total": fr.total, "correct": fr.correct, "abstain": fr.abstain, "wrong": fr.wrong, "pa": round(fr.penalized_accuracy, 6)}
    result["by_format"] = by_format
    return result


# Trong Phiên score_only, tự tìm thư mục kết quả đã khóa từ Phiên inference nếu EVAL_SUBDIRS trống
if EXECUTION_STAGE == "score_only" and not EVAL_SUBDIRS:
    score_base = Path(SCORING_INPUT_RUN_DIR) if SCORING_INPUT_RUN_DIR else None
    if score_base is None:
        score_base = find_candidate_dir(lambda p: any(p.glob("*/predictions.jsonl")))
    if score_base and score_base.is_dir():
        if (score_base / "predictions.jsonl").is_file() and (score_base / "manifest.json").is_file():
            EVAL_SUBDIRS = [score_base]
        else:
            EVAL_SUBDIRS = sorted(
                p for p in score_base.iterdir()
                if p.is_dir() and (p / "predictions.jsonl").is_file() and (p / "manifest.json").is_file()
            )
    print(f"Found {len(EVAL_SUBDIRS)} run dirs for score_only evaluation.")

REF_MAP = None
if EXECUTION_STAGE == "score_only" and REFERENCE_PATH and REFERENCE_PATH.is_file():
    REF_MAP = {}
    for ln in REFERENCE_PATH.read_text(encoding="utf-8-sig").splitlines():
        if ln.strip():
            rec = json.loads(ln)
            REF_MAP[str(rec["id"])] = rec
    print(f"Loaded {len(REF_MAP)} reference records from {REFERENCE_PATH}")
elif EXECUTION_STAGE == "inference":
    print("Phiên inference đã xuất predictions & trace vào RUN_DIR. Để chấm PA an toàn, attach predictions + reference và chạy lại với EXECUTION_STAGE='score_only'.")

SUMMARY_REPORTS = []
for sub_dir in EVAL_SUBDIRS:
    rep = score_run_dir(sub_dir, REF_MAP)
    if EXECUTION_STAGE == "score_only" or (sub_dir / "manifest.json").is_file():
        out_rep_dir = RUN_DIR if EXECUTION_STAGE == "score_only" else sub_dir
        (out_rep_dir / f"{sub_dir.name}_evaluation_report.json" if EXECUTION_STAGE == "score_only" else out_rep_dir / "evaluation_report.json").write_text(
            json.dumps(rep, indent=2, ensure_ascii=False), encoding="utf-8"
        )
    SUMMARY_REPORTS.append(rep)

(RUN_DIR / "summary_report.json").write_text(json.dumps({
    "run_dir": str(RUN_DIR),
    "execution_stage": EXECUTION_STAGE,
    "runtime_info": RUNTIME_INFO,
    "smoke_results": SMOKE_RESULTS,
    "runs": SUMMARY_REPORTS,
}, indent=2, ensure_ascii=False), encoding="utf-8")

if SUMMARY_REPORTS:
    print("\n==================== EVALUATION SUMMARY ====================")
    for rep in SUMMARY_REPORTS:
        print(
            f"[{rep['run_label']}] mode={rep['tool_mode']} temp={rep['temperature']} | "
            f"N={rep.get('scored_total', rep['question_count'])} | "
            f"Correct={rep.get('correct', '-')} | Abstain={rep.get('abstain', '-')} | Wrong={rep.get('wrong', '-')} | "
            f"PA={rep.get('pa', 'N/A')} | RescuePA={rep.get('shadow_tool_rescue', {}).get('pa', 'N/A')} | "
            f"tool_ok_empty_final={rep['tool_succeeded_but_final_empty']} | time={rep['total_elapsed_sec']}s"
        )
        if rep.get("by_format"):
            print("   by_format  :", rep["by_format"])
        print("   type_source:", rep["type_sources"])
